## Multi Agentic Room Furnishing Recommender v28

## 1. Setup Install Dependencies

In [ ]:
!pip install -q langgraph groq requests matplotlib pandas google-search-results

In [ ]:
!pip install -q huggingface_hub

## 2. Import Library

In [ ]:
import os
import re
import json
import time
import getpass
import random
import threading
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import TypedDict, List, Dict, Any, Optional

from groq import Groq
from langgraph.graph import StateGraph, END

print("Library siap. (Tidak perlu GPU untuk sistem versi layout-based ini.)")

## 3. Groq LLM Client

In [ ]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    print("Secret 'GROQ_API_KEY' belum ditemukan, masukkan manual (tidak akan tersimpan).")
    GROQ_API_KEY = getpass.getpass("Masukkan Groq API key: ").strip()

if not GROQ_API_KEY:
    raise RuntimeError(
        "GROQ_API_KEY kosong. Sistem butuh key ini untuk Core Furniture & Theme/Decor Agent. "
        "Daftar gratis (tanpa kartu kredit) di https://console.groq.com lalu jalankan ulang cell ini."
    )

groq_client = Groq(api_key=GROQ_API_KEY)

GROQ_MODEL_CANDIDATES = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
]

GROQ_MAX_RETRIES = 3
GROQ_BASE_BACKOFF_S = 2

class GroqUnavailableError(RuntimeError):
    pass

def _is_auth_error(e: Exception) -> bool:
    status = getattr(e, "status_code", None)
    msg = str(e).lower()
    return status in (401, 403) or "invalid api key" in msg or "unauthorized" in msg

def _is_rate_limit_error(e: Exception) -> bool:
    status = getattr(e, "status_code", None)
    return status == 429 or "rate_limit" in str(e).lower() or "429" in str(e)

def _is_model_unavailable_error(e: Exception) -> bool:
    status = getattr(e, "status_code", None)
    msg = str(e).lower()
    return status == 404 or "model_not_found" in msg or "decommissioned" in msg or "does not exist" in msg

def _extract_json_list(text: str):
    text = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    data = json.loads(text)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ("items", "data", "result", "recommendations"):
            if key in data and isinstance(data[key], list):
                return data[key]
        return [data]
    raise ValueError(f"Struktur JSON tidak dikenali: {type(data)}")

def ask_llm_json(prompt: str, retries: int = GROQ_MAX_RETRIES):
    last_err = None
    for model in GROQ_MODEL_CANDIDATES:
        for attempt in range(retries + 1):
            try:
                resp = groq_client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "Kamu selalu menjawab HANYA dengan JSON valid, tanpa teks lain, tanpa markdown code fence."},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.4,
                )
            except Exception as e:
                last_err = e
                if _is_auth_error(e):
                    print(f"  [error] Groq API key sepertinya tidak valid/tidak diotorisasi: {e}")
                    raise GroqUnavailableError(f"API key Groq tidak valid: {e}") from e
                if _is_model_unavailable_error(e):
                    print(f"  [warn] Model '{model}' tidak tersedia ({e}). Pindah ke model kandidat berikutnya...")
                    break
                rate_limited = _is_rate_limit_error(e)
                if attempt == retries:
                    print(f"  [warn] Model '{model}' gagal setelah {retries + 1} percobaan: {e}. "
                          f"Coba model kandidat berikutnya (kalau ada)...")
                    break
                wait = GROQ_BASE_BACKOFF_S * (2 ** attempt) * (2 if rate_limited else 1)
                tag = "rate limit" if rate_limited else "error sementara"
                print(f"  [warn] Groq {tag} di model '{model}' (percobaan {attempt + 1}/{retries + 1}): {e}. Tunggu {wait}s...")
                time.sleep(wait)
                continue

            text = resp.choices[0].message.content.strip()
            try:
                return _extract_json_list(text)
            except (json.JSONDecodeError, ValueError) as e:
                last_err = e
                if attempt == retries:
                    print(f"  [warn] Model '{model}' mengembalikan JSON tidak valid setelah {retries + 1} percobaan. "
                          f"Coba model kandidat berikutnya (kalau ada)...")
                    break
                time.sleep(1)
                continue

    raise GroqUnavailableError(
        f"Semua model kandidat Groq ({', '.join(GROQ_MODEL_CANDIDATES)}) gagal dipanggil. Error terakhir: {last_err}"
    )

print(f"Groq client siap. Model kandidat (urut prioritas): {GROQ_MODEL_CANDIDATES}.")
print("Retry + backoff khusus rate-limit + fallback antar-model + deteksi API key salah aktif.")

try:
    from google.colab import userdata as _ud
    HF_TOKEN = _ud.get("hungging-room-planner")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    print("ℹ️  HF_TOKEN tidak ditemukan. Masukkan manual (opsional).")
    _hf_input = getpass.getpass("Hugging Face token (atau Enter untuk skip): ").strip()
    HF_TOKEN = _hf_input if _hf_input else None

if HF_TOKEN:
    print("✅ HF_TOKEN ditemukan — generate gambar interior 3D aktif.")
else:
    print("⚠️  HF_TOKEN kosong — gambar 3D dinonaktifkan (floor plan 2D tetap jalan).")

## 4. State Schema LangGraph

In [ ]:
class RoomPlannerState(TypedDict):
    room_type: str
    room_length_m: float
    room_width_m: float
    room_height_m: float
    style_preference: str
    user_budget_total: int

    layout_plan: Dict[str, Any]
    core_items: List[Dict[str, Any]]
    decor_items: List[Dict[str, Any]]
    recommended_items: List[Dict[str, Any]]

    search_results_batch_a: Dict[str, List[Dict[str, Any]]]
    search_results_batch_b: Dict[str, List[Dict[str, Any]]]
    search_results: Dict[str, List[Dict[str, Any]]]

    selected_items: List[Dict[str, Any]]
    placement: List[Dict[str, Any]]
    ceiling_items: List[Dict[str, Any]]
    final_floorplan_path: str

    retry_count: Dict[str, int]
    retry_modifiers: Dict[str, str]
    _needs_retry: bool

    floor_number: int
    windows: List[Dict[str, Any]]
    custom_items: List[Dict[str, Any]]
    review_results: Dict[str, Any]
    budget_target_pct: float

    user_location: str
    tukang_results: Dict[str, List[Dict[str, Any]]]

    generated_image_path: str

## 5. Node 1 Layout Planner Agent

In [ ]:
WALKWAY_CLEARANCE_M = 0.6
WALL_ZONE_DEPTH_M = 0.8
DOOR_WIDTH_M = 0.9
DOOR_CLEARANCE_M = 0.9

def layout_planner_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Layout Planner Agent] menghitung tata letak dari ukuran ruangan...")
    L, W, H = state["room_length_m"], state["room_width_m"], state["room_height_m"]
    total_area = round(L * W, 2)

    door_center_x = L / 2
    door_zone = {
        "x_min": max(0, door_center_x - DOOR_WIDTH_M / 2 - 0.15),
        "x_max": min(L, door_center_x + DOOR_WIDTH_M / 2 + 0.15),
        "y_min": 0,
        "y_max": DOOR_CLEARANCE_M,
    }

    wall_zones = [
        {"id": "dinding_bawah", "x_min": 0, "x_max": L, "y_min": 0, "y_max": min(WALL_ZONE_DEPTH_M, W)},
        {"id": "dinding_atas", "x_min": 0, "x_max": L, "y_min": max(0, W - WALL_ZONE_DEPTH_M), "y_max": W},
        {"id": "dinding_kiri", "x_min": 0, "x_max": min(WALL_ZONE_DEPTH_M, L), "y_min": 0, "y_max": W},
        {"id": "dinding_kanan", "x_min": max(0, L - WALL_ZONE_DEPTH_M), "x_max": L, "y_min": 0, "y_max": W},
    ]

    usable_area = round(total_area * 0.65, 2)

    layout_plan = {
        "room_type": state["room_type"],
        "length_m": L,
        "width_m": W,
        "height_m": H,
        "total_area_m2": total_area,
        "usable_area_m2": usable_area,
        "door_zone": door_zone,
        "wall_zones": wall_zones,
        "walkway_clearance_m": WALKWAY_CLEARANCE_M,
        "windows": state.get("windows", []),
        "notes": (
            f"Ruangan {L}m x {W}m x {H}m ({total_area} m2). "
            f"Furnitur besar disarankan menempel dinding, sisakan jalur sirkulasi "
            f"min {WALKWAY_CLEARANCE_M} m dan area depan pintu tetap kosong."
        ),
    }

    print(f"  Luas total: {total_area} m2 | Luas efektif untuk furnitur: {usable_area} m2")
    state["layout_plan"] = layout_plan
    return state

## 5b. Cek Kompatibilitas Lantai

In [ ]:
import functools

@functools.lru_cache(maxsize=128)
def _llm_floor_check(category: str, floor_number: int, width_cm: int, depth_cm: int, height_cm: int) -> dict:
    prompt = f"""Kamu adalah konsultan logistik Indonesia.
Item: "{category}", dimensi {width_cm}x{depth_cm}x{height_cm} cm, lantai {floor_number}.

Balas HANYA JSON (tanpa teks lain):
{{"compatible": true/false, "weight_est_kg": <angka>, "can_be_disassembled": true/false, "note": "<catatan MAKS 50 KARAKTER>"}}

Aturan note: singkat saja, contoh: "✅ Aman ~15 kg" atau "⚠️ Berat ~80 kg, minta jasa angkut" """

    try:
        raw = None
        for _model in GROQ_MODEL_CANDIDATES:
            try:
                resp = groq_client.chat.completions.create(
                    model=_model,
                    messages=[
                        {"role": "system", "content": "Jawab HANYA dengan JSON valid, tanpa teks lain."},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.1,
                    max_tokens=150,
                )
                raw = resp.choices[0].message.content.strip()
                break
            except Exception:
                continue
        if raw is None:
            raise RuntimeError("Semua model gagal")
        import re as _re
        raw = _re.sub(r"```(?:json)?|```", "", raw).strip()
        result = json.loads(raw)
        return {
            "compatible":          bool(result.get("compatible", True)),
            "weight_est_kg":       float(result.get("weight_est_kg", 20)),
            "can_be_disassembled": bool(result.get("can_be_disassembled", True)),
            "note":                str(result.get("note", "✅ OK"))[:60],
        }
    except Exception as e:
        _heavy = any(k in category.lower() for k in ["bathtub","jacuzzi","piano","granit","beton"])
        _w = float(width_cm) / 100
        _d = float(depth_cm) / 100
        _h = float(height_cm) / 100
        _kg = _w * _d * _h * 18
        _ok = _kg < 60 or not _heavy
        return {
            "compatible":          _ok or floor_number <= 1,
            "weight_est_kg":       _kg,
            "can_be_disassembled": not _heavy,
            "note":                f"{'✅' if _ok else '⚠️'} ~{_kg:.0f} kg (estimasi rule-based, LLM gagal: {e})",
        }

def check_floor_compatibility(item_spec: dict, floor_number: int) -> dict:
    if floor_number <= 1:
        return {"compatible": True, "note": "Lantai 1 — semua item OK",
                "can_be_disassembled": True, "weight_est_kg": 0}
    return _llm_floor_check(
        category    = str(item_spec.get("category", "item")),
        floor_number= int(floor_number),
        width_cm    = int(item_spec.get("suggested_width_cm", 60)),
        depth_cm    = int(item_spec.get("suggested_depth_cm", 40)),
        height_cm   = int(item_spec.get("suggested_height_cm", 60)),
    )

print("Floor compatibility checker (LLM-based) siap.")

## 5c. Custom Item Agent

In [ ]:
CUSTOM_ITEM_TYPES_EMERGENCY_FALLBACK = {
    "kabinet built-in":  {"tukang": "tukang kayu/meubel", "material_utama": ["papan MDF", "engsel", "cat duco"]},
    "rak dinding tanam": {"tukang": "tukang kayu",        "material_utama": ["papan kayu/MDF", "bracket", "cat"]},
    "plafon gypsum":     {"tukang": "tukang gypsum",      "material_utama": ["rangka hollow", "gypsum board", "cat"]},
    "partisi dinding":   {"tukang": "tukang bata/gypsum", "material_utama": ["bata ringan", "gypsum board", "cat"]},
    "pintu geser":       {"tukang": "tukang kayu/aluminium", "material_utama": ["rel pintu", "handle", "daun pintu"]},
}

print("Custom item: mode 100% LLM. Emergency fallback tersedia jika LLM mati total.")

## 6. Node 2a 2b Core Furniture Theme Decor Agent

In [ ]:
def _recommendation_prompt(state: RoomPlannerState, focus: str, item_count: str, extra_rule: str) -> str:
    lp = state["layout_plan"]
    budget_total  = state["user_budget_total"]
    budget_target = int(budget_total * state.get("budget_target_pct", 0.80))
    return f'''Kamu adalah interior designer AI. Rancang rekomendasi {focus} untuk SATU ruangan
berikut (rancang seolah ruangan ini kosong total, belum ada furnitur apapun):

- Jenis ruangan: {state["room_type"]}
- Ukuran ruangan: {lp["length_m"]} m x {lp["width_m"]} m, tinggi plafon {lp["height_m"]} m
- Luas total: {lp["total_area_m2"]} m2, luas efektif untuk furnitur: {lp["usable_area_m2"]} m2
- Preferensi gaya/tema user: {state.get("style_preference") or "tidak ada, pilih gaya netral yang nyaman"}
- Budget total user   : Rp{budget_total:,}
- Target pemakaian    : Rp{budget_target:,} (harus mendekati angka ini, jangan jauh di bawahnya)

Aturan:
1. Usulkan {item_count} yang MASUK AKAL untuk "{state["room_type"]}" (jenis ruangan ini).
   PENTING: semua item HARUS relevan dengan {state["room_type"]} — jangan rekomendasikan
   item untuk ruangan lain (misal kasur/lemari tidak relevan untuk dapur).
   Untuk dapur → kompor, sink, rak bumbu, kitchen set, dll.
   Untuk kamar tidur → kasur, lemari pakaian, nakas, dll.
   Jangan usulkan barang yang jelas tidak muat (mis. kasur king di ruangan 2x2 m).
2. Setiap item harus barang yang benar-benar dijual di Tokopedia/Shopee (barang nyata, bukan custom project).
3. {extra_rule}
4. PENTING: Estimated_price harus realistis dan proporsional.
   Total semua estimated_price harus mendekati Rp{budget_target:,}.
   Pembagian kasar per item = Rp{budget_target:,} / jumlah item.
   Item utama (kasur, meja makan, sofa) boleh 2-3x dari rata-rata.
   Item kecil (aksesoris, hiasan) boleh 0.2-0.5x dari rata-rata.
   WAJIB: sesuaikan kualitas produk dengan budget — kalau budget item Rp3jt+,
   rekomendasikan produk kualitas mid-range, bukan yang paling murah.
5. Jawab HANYA dalam format JSON list, setiap elemen object dengan field persis berikut:
   "category" (nama kategori singkat, contoh: "Kasur"),
   "reasoning" (alasan singkat kenapa cocok),
   "search_query" (kata kunci pencarian marketplace yang efektif — WAJIB sertakan nama ruangan, contoh: "lemari dapur minimalis" bukan cuma "lemari"),
   "estimated_price" (integer rupiah, estimasi harga wajar),
   "suggested_width_cm" (integer),
   "suggested_depth_cm" (integer),
   "suggested_height_cm" (integer),
   "placement" ("dinding" jika menempel dinding, "lantai" jika freestanding/tengah ruangan,
   atau "langit-langit" KHUSUS barang yang digantung/dipasang di plafon seperti lampu gantung
   -- barang ini TIDAK menempati ruang lantai),
   "priority" (integer 1 = paling wajib, makin besar makin opsional)
'''

def _safe_int(value, default: int) -> int:
    try:
        result = int(float(value))
        return result if result > 0 else default
    except (TypeError, ValueError):
        return default

def _sanitize_items(raw_items, max_count: int, default_placement: str):
    if not isinstance(raw_items, list):
        return []
    cleaned = []
    for raw in raw_items:
        if not isinstance(raw, dict) or not str(raw.get("category") or "").strip():
            continue
        item = {
            "category": str(raw["category"]).strip()[:60],
            "reasoning": str(raw.get("reasoning", "")).strip()[:200],
            "search_query": str(raw.get("search_query") or raw["category"]).strip()[:80],
            "estimated_price": _safe_int(raw.get("estimated_price"), 300000),
            "suggested_width_cm": _safe_int(raw.get("suggested_width_cm"), 60),
            "suggested_depth_cm": _safe_int(raw.get("suggested_depth_cm"), 60),
            "suggested_height_cm": _safe_int(raw.get("suggested_height_cm"), 60),
            "placement": raw.get("placement") if raw.get("placement") in ("dinding", "lantai", "langit-langit") else default_placement,
            "priority": _safe_int(raw.get("priority"), 5),
        }
        cleaned.append(item)
        if len(cleaned) >= max_count:
            break
    return cleaned

def _llm_room_items(state: RoomPlannerState, focus: str, max_count: int, default_placement: str) -> list:
    lp = state.get("layout_plan") or {}
    rt = state.get("room_type", "ruangan umum")
    budget = state.get("user_budget_total", 2_000_000)
    L = lp.get("length_m", 3)
    W = lp.get("width_m", 3)
    prompt = f"""Kamu adalah interior designer Indonesia.
Ruangan: {rt}, ukuran {L}x{W}m, budget Rp{budget:,}.
Usulkan {max_count} {focus} yang cocok untuk ruangan ini.
Jawab HANYA JSON list. Setiap item punya field:
"category","search_query","estimated_price","suggested_width_cm","suggested_depth_cm","suggested_height_cm","placement","priority","reasoning"
placement harus salah satu: "dinding","lantai","langit-langit"."""
    try:
        raw = ask_llm_json(prompt)
        items = _sanitize_items(raw, max_count=max_count, default_placement=default_placement)
        if items:
            return items
    except Exception as e:
        print(f"  [fallback LLM juga gagal: {e}]")
    return _sanitize_items([
        {"category": f"Furnitur {rt}", "search_query": f"furnitur {rt} minimalis",
         "estimated_price": max(150_000, budget // 5),
         "suggested_width_cm": 80, "suggested_depth_cm": 50, "suggested_height_cm": 80,
         "placement": default_placement, "priority": 1,
         "reasoning": f"Emergency fallback: Groq gagal total saat run ini untuk {rt}."},
    ], max_count=1, default_placement=default_placement)

def _fallback_core_items(state: RoomPlannerState):
    print("  [fallback] Groq gagal pada attempt pertama — coba prompt sederhana per ruangan...")
    return _llm_room_items(state, focus="furnitur fungsional wajib (bukan dekorasi)",
                           max_count=4, default_placement="dinding")

def _fallback_decor_items(state: RoomPlannerState):
    print("  [fallback] Groq gagal pada attempt pertama — coba prompt sederhana per ruangan...")
    return _llm_room_items(state, focus="item dekoratif/aksesoris sesuai tema",
                           max_count=3, default_placement="lantai")

def core_furniture_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Core Furniture Agent] menyusun furnitur fungsional wajib (paralel dengan Theme/Decor Agent)...")
    prompt = _recommendation_prompt(
        state, focus="furnitur FUNGSIONAL WAJIB (bukan dekorasi)", item_count="6 kategori furnitur inti (wajib ada, sesuai jenis ruangan)",
        extra_rule=("Fokus ke kebutuhan dasar ruangan ini agar layak huni/dipakai (jangan usulkan item dekoratif murni). "
                    "KHUSUS ruang makan: wajib ada meja makan DAN kursi makan 4-6pcs — gunakan "
                    "search_query seperti 'set kursi makan 4 kursi' atau 'kursi makan minimalis 4pcs'. "
                    "KHUSUS dapur: wajib ada kompor dan sink/wastafel. "
                    "KHUSUS kamar tidur: wajib ada kasur. "
                    "Pastikan search_query menyertakan nama ruangan."),
    )
    try:
        raw_items = ask_llm_json(prompt)
        items = _sanitize_items(raw_items, max_count=4, default_placement="dinding")
        if not items:
            raise ValueError("LLM mengembalikan list kosong/tidak valid setelah sanitasi.")
    except Exception as e:
        print(f"  [warn] Core Furniture Agent gagal pakai LLM ({e}).")
        items = _fallback_core_items(state)
    for it in items:
        it["source_agent"] = "core_furniture"
    for it in items:
        print(f"  - {it['category']} (~Rp{it['estimated_price']:,}, "
              f"{it['suggested_width_cm']}x{it['suggested_depth_cm']}x{it['suggested_height_cm']} cm)")
    return {"core_items": items}

def _room_type_filler_hint(room_type: str) -> str:
    rt = room_type.lower()
    if any(k in rt for k in ("tamu", "keluarga", "santai", "nonton")):
        return "televisi + rak TV, akuarium kecil, karpet, bantal sofa, lukisan/hiasan dinding, tanaman hias, jam dinding"
    if "tidur" in rt or "kamar anak" in rt or "nursery" in rt:
        return "lampu tidur, jam dinding, tanaman hias, cermin, hiasan dinding, karpet kecil"
    if "makan" in rt:
        return "lampu gantung meja makan, taplak/runner meja, hiasan dinding, tanaman hias meja"
    if "kerja" in rt:
        return "lampu meja, rak buku kecil, tanaman hias meja, jam dinding, organizer meja"
    if "dapur" in rt:
        return "rak bumbu, jam dinding dapur, tanaman herbal kecil, hiasan dinding dapur"
    if "mandi" in rt:
        return "rak handuk, cermin, keset kamar mandi, tanaman kecil tahan lembap"
    return "hiasan dinding, tanaman hias, karpet, jam dinding, lampu hias"

def theme_decor_node(state: RoomPlannerState) -> RoomPlannerState:
    print("[Theme/Decor Agent] menyusun item dekoratif sesuai tema (paralel dengan Core Furniture Agent)...")
    prompt = _recommendation_prompt(
        state, focus="item DEKORATIF/AKSESORIS/PENGISI RUANGAN sesuai tema user", item_count="4 kategori item dekoratif/pengisi",
        extra_rule=(
            "Fokus ke ekspresi tema/preferensi user pada ruangan (dekorasi, pencahayaan, pajangan), "
            "tetap barang nyata dan proporsional untuk ukuran ruangan ini. "
            "PENTING: prioritaskan item PENGISI berukuran KECIL-SEDANG (BUKAN furnitur besar seperti "
            "sofa/meja/lemari -- itu sudah dihandle Core Furniture Agent), supaya ruangan tidak terasa "
            f"kosong. Contoh yang relevan untuk ruangan ini: {_room_type_filler_hint(state['room_type'])}. "
            "Sesuaikan dengan jenis ruangan yang sebenarnya, jangan asal contek daftar contoh kalau tidak relevan."
        ),
    )
    try:
        raw_items = ask_llm_json(prompt)
        items = _sanitize_items(raw_items, max_count=3, default_placement="lantai")
        if not items:
            raise ValueError("LLM mengembalikan list kosong/tidak valid setelah sanitasi.")
    except Exception as e:
        print(f"  [warn] Theme/Decor Agent gagal pakai LLM ({e}).")
        items = _fallback_decor_items(state)
    for it in items:
        it["source_agent"] = "theme_decor"
    for it in items:
        print(f"  - {it['category']} (~Rp{it['estimated_price']:,}, "
              f"{it['suggested_width_cm']}x{it['suggested_depth_cm']}x{it['suggested_height_cm']} cm)")
    return {"decor_items": items}

## 6c. Node 2c Recommendation Merge Agent

In [ ]:
def _normalize_words(text: str):
    return set(re.sub(r"[^a-z0-9\s]", " ", text.lower()).split())

def _is_probable_duplicate(cat_a: str, cat_b: str, ignore_words: set = frozenset()) -> bool:
    wa, wb = _normalize_words(cat_a) - ignore_words, _normalize_words(cat_b) - ignore_words
    if not wa or not wb:
        return False
    overlap = wa & wb
    return len(overlap) / min(len(wa), len(wb)) >= 0.5

def recommendation_merge_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Recommendation Merge Agent] menggabungkan furnitur inti + item tematik, "
          "validasi kapasitas & duplikasi...")
    core = state.get("core_items", [])
    decor = state.get("decor_items", [])
    for it in decor:
        it["priority"] = it.get("priority", 5) + 10
    candidates = sorted(core + decor, key=lambda it: it.get("priority", 99))
    if not candidates:
        print("  [warn] Tidak ada kategori furnitur yang berhasil diusulkan sama sekali "
              "(Core & Theme/Decor Agent kosong).")

    lp = state["layout_plan"]
    remaining_area_m2 = lp["usable_area_m2"]

    room_type_words = _normalize_words(state.get("room_type", ""))
    accepted, skipped = [], []
    for item in candidates:
        cat = item["category"]

        if any(_is_probable_duplicate(cat, a["category"], ignore_words=room_type_words) for a in accepted):
            item["skip_reason"] = "duplikat_kategori"
            skipped.append(item)
            continue

        w_m = (item.get("suggested_width_cm") or 0) / 100
        d_m = (item.get("suggested_depth_cm") or 0) / 100
        footprint_m2 = round(w_m * d_m, 3)
        if accepted and footprint_m2 > remaining_area_m2:
            item["skip_reason"] = "melebihi_kapasitas_ruangan"
            skipped.append(item)
            continue

        remaining_area_m2 = max(0.0, round(remaining_area_m2 - footprint_m2, 3))
        accepted.append(item)

    state["recommended_items"] = accepted
    print(f"  Diterima {len(accepted)} kategori (dari {len(core)} fungsional + {len(decor)} dekoratif diusulkan)")
    for s in skipped:
        print(f"  - dilewati: {s['category']} ({s['skip_reason']})")
    return state

## 6d. Node 2d Item Selection Agent

In [ ]:
def _format_rupiah(n: int) -> str:
    return f"Rp{int(n):,}"

def _show_item_checklist(items, header: str = ""):
    if header:
        print(header)
    for i, it in enumerate(items, start=1):
        reason = (it.get("reasoning", "") or "")[:55]
        print(f"  [{i}] {it['category']:<28} ~{_format_rupiah(it['estimated_price']):>14}"
              f"  ({reason})")

def _ask_exclude_indices(n_items: int) -> set:
    raw = input("  Nomor barang yang MAU DIHAPUS dari daftar (pisahkan koma, contoh: 2,5). "
                "Kosongkan kalau mau pertahankan semua: ").strip()
    if not raw:
        return set()
    out = set()
    for tok in raw.split(","):
        tok = tok.strip()
        if tok.isdigit():
            idx = int(tok)
            if 1 <= idx <= n_items:
                out.add(idx)
    return out

def _llm_topup_wishlist(state: RoomPlannerState, existing_categories: list,
                         need_amount: int, max_count: int = 3,
                         removed_categories: list = None) -> list:
    rt = state["room_type"]
    lp = state["layout_plan"]
    hint = _room_type_filler_hint(rt)
    exist_str = ", ".join(existing_categories) or "(belum ada)"
    removed_note = ""
    if removed_categories:
        removed_note = (f"\nUser BARU SAJA menghapus: {', '.join(removed_categories)} -- coba usulkan "
                         f"PENGGANTI yang mengisi fungsi/kesan serupa (tidak harus identik), "
                         f"selain opsi pengisi umum lainnya.\n")
    prompt = f"""Kamu interior designer AI. Ruangan "{rt}" ukuran {lp['length_m']}x{lp['width_m']}m
sudah direncanakan berisi kategori barang berikut: {exist_str}.
{removed_note}
Sisa budget yang belum terpakai dari rencana di atas: Rp{need_amount:,}. Supaya ruangan tidak
terasa kosong, usulkan {max_count} item PENGISI TAMBAHAN berukuran kecil-sedang (BUKAN
pengulangan kategori yang sudah ada di atas, dan BUKAN furnitur besar seperti sofa/lemari/kasur),
misalnya seperti: {hint}. Sesuaikan relevansi dengan jenis ruangan "{rt}" yang sebenarnya.

Jawab HANYA dalam format JSON list, tiap elemen object dengan field persis:
"category","reasoning","search_query","estimated_price","suggested_width_cm",
"suggested_depth_cm","suggested_height_cm","placement" ("dinding"/"lantai"/"langit-langit"),
"priority" (integer 15-25, ini item tambahan bukan utama).
Total estimated_price semua item idealnya mendekati Rp{need_amount:,}, jangan jauh melebihi."""
    try:
        raw = ask_llm_json(prompt)
        items = _sanitize_items(raw, max_count=max_count, default_placement="lantai")
        for it in items:
            it["source_agent"] = "wishlist_topup"
        return items
    except Exception as e:
        print(f"  [warn] Gagal minta usulan tambahan dari LLM: {e}")
        return []

def item_selection_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Item Selection Agent] daftar rencana barang untuk ruangan ini "
          "(sebelum dicari link produk & tukangnya):")
    items = list(state.get("recommended_items", []))
    budget_total  = state["user_budget_total"]
    budget_target = int(budget_total * state.get("budget_target_pct", 0.80))

    round_no = 1
    while True:
        _show_item_checklist(
            items,
            header=f"\n  -- Daftar barang (update ke-{round_no}) --" if round_no > 1 else "",
        )
        current_total = sum(it["estimated_price"] for it in items)
        print(f"  Estimasi total saat ini: {_format_rupiah(current_total)} "
              f"dari target pemakaian {_format_rupiah(budget_target)}")

        excl = _ask_exclude_indices(len(items))
        replaced_this_round = False
        if excl:
            removed_specs = [it for i, it in enumerate(items, start=1) if i in excl]
            removed = [it["category"] for it in removed_specs]
            removed_total = sum(it.get("estimated_price", 0) for it in removed_specs)
            items = [it for i, it in enumerate(items, start=1) if i not in excl]
            print(f"  ❌ Dihapus dari daftar: {', '.join(removed)}")

            ans_repl = input(f"  Mau AI usulkan barang pengganti buat mengisi slot yang baru "
                              f"dihapus ({', '.join(removed)})? (ya/tidak): ").strip().lower()
            if ans_repl in ("ya", "y", "yes"):
                existing_cats = [it["category"] for it in items]
                repl_amount = removed_total if removed_total > 0 else int(budget_target * 0.1)
                replacement = _llm_topup_wishlist(state, existing_cats, repl_amount,
                                                   max_count=max(1, min(len(removed), 3)),
                                                   removed_categories=removed)
                if replacement:
                    print(f"  ✅ LLM mengusulkan {len(replacement)} barang pengganti:")
                    items.extend(replacement)
                    replaced_this_round = True
                else:
                    print("  (Tidak ada usulan pengganti yang berhasil dibuat.)")

        if replaced_this_round:
            round_no += 1
            continue

        current_total = sum(it["estimated_price"] for it in items)
        gap = budget_target - current_total

        if gap > budget_target * 0.25 and gap > 200_000:
            print(f"\n  💡 Estimasi baru pakai {_format_rupiah(current_total)}, masih ada sisa "
                  f"{_format_rupiah(gap)} dari target -- ruangan berisiko terasa kosong.")
            ans = input("  Mau AI usulkan barang pengisi tambahan? (ya/tidak): ").strip().lower()
            if ans in ("ya", "y", "yes"):
                existing_cats = [it["category"] for it in items]
                extra = _llm_topup_wishlist(state, existing_cats, gap)
                if extra:
                    print(f"  ✅ LLM mengusulkan {len(extra)} barang pengisi tambahan:")
                    items.extend(extra)
                    round_no += 1
                    continue
                else:
                    print("  (Tidak ada usulan tambahan yang berhasil dibuat, lanjut apa adanya.)")
        break

    state["recommended_items"] = items
    final_total = sum(it["estimated_price"] for it in items)
    print(f"\n  ✅ Daftar final: {len(items)} kategori barang, estimasi total "
          f"{_format_rupiah(final_total)}. Lanjut cari link produk asli & tukang...")
    return state

print("Item Selection Agent siap.")

## 7. Node 3a 3b Product Search Agent

In [ ]:
try:
    from google.colab import userdata
    SERPAPI_KEY = userdata.get("SERPAPI_KEY")
except Exception:
    SERPAPI_KEY = None

if not SERPAPI_KEY:
    import getpass as _gp
    print("Secret 'SERPAPI_KEY' belum ditemukan, masukkan manual.")
    SERPAPI_KEY = _gp.getpass("Masukkan SerpApi API key: ").strip()

if not SERPAPI_KEY:
    print("[warn] SERPAPI_KEY kosong -- search agent pakai LLM catalog fallback (Tokopedia/Shopee link nyata).")
else:
    try:
        from serpapi import GoogleSearch as _TestSearch
        _test = _TestSearch({"engine": "google_shopping", "q": "meja", "api_key": SERPAPI_KEY,
                             "gl": "id", "hl": "id", "num": 3}).get_dict()
        _test_err = _test.get("error")
        if _test_err:
            print(f"[warn] SERPAPI_KEY kelihatannya BERMASALAH: {_test_err}")
            print("       -> Semua barang akan jatuh ke LLM catalog fallback (belum terverifikasi) sampai ini dibenerin.")
            print("       -> Cek key/kuota di https://serpapi.com/manage-api-key")
        elif _test.get("shopping_results"):
            print(f"[ok] SERPAPI_KEY valid -- tes query 'meja' balik {len(_test['shopping_results'])} hasil.")
        else:
            print("[warn] SERPAPI_KEY jalan tapi tes query 'meja' balik 0 hasil (aneh, biasanya ada). Lanjut jalan, tapi pantau warning SerpApi berikutnya.")
    except Exception as _e:
        print(f"[warn] Tes SERPAPI_KEY gagal: {_e}")
        print("       -> Semua barang akan jatuh ke LLM catalog fallback (belum terverifikasi) sampai ini dibenerin.")

REQUEST_JITTER_RANGE_S = (0.1, 0.3)
SEARCH_RESULT_LIMIT = 15
MAX_CONSECUTIVE_FAILURES = 6
SEARCH_MAX_WORKERS = 4

_failure_lock = threading.Lock()

def _effective_query(item: Dict[str, Any], state: RoomPlannerState) -> str:
    base_query  = item["search_query"]
    room_type   = state.get("room_type", "")
    est_price   = item.get("estimated_price", 0)
    strategy    = state.get("retry_modifiers", {}).get(item["category"])
    attempt     = state.get("retry_count", {}).get(item["category"], 0)

    def _price_hint(p: int) -> str:
        if p <= 0:    return ""
        if p < 200_000:   return "murah"
        if p < 500_000:   return "terjangkau"
        if p < 1_500_000: return "harga menengah"
        if p < 5_000_000: return "premium"
        return "high-end"

    price_hint = _price_hint(est_price)

    if strategy == "simplify":
        if attempt <= 1:
            q = f"{item['category']} {room_type}".strip()
        else:
            q = f"{item['category']} {price_hint}".strip()
        return q
    if strategy == "smaller_cheaper":
        return f"{base_query} minimalis kecil murah".strip()

    if price_hint and price_hint not in base_query.lower():
        return f"{base_query} {price_hint}".strip()
    return base_query

def _circuit_open(failure_counters: Dict[str, int]) -> bool:
    return failure_counters.get("global", 0) >= MAX_CONSECUTIVE_FAILURES

def _record_result(failure_counters: Dict[str, int], success: bool) -> None:
    with _failure_lock:
        if success:
            failure_counters["global"] = 0
        else:
            failure_counters["global"] = failure_counters.get("global", 0) + 1

def _parse_price_idr(price_str: str) -> str:
    if not price_str:
        return "Rp-"
    cleaned = re.sub(r"[^0-9]", "", price_str)
    return f"Rp{int(cleaned):,}" if cleaned else price_str

def _try_serpapi_engine(engine: str, params_extra: dict,
                         failure_counters: Dict[str, int], query: str) -> List[Dict[str, Any]]:
    if not SERPAPI_KEY or _circuit_open(failure_counters):
        return []
    time.sleep(random.uniform(*REQUEST_JITTER_RANGE_S))
    try:
        from serpapi import GoogleSearch
        params = {"engine": engine, "api_key": SERPAPI_KEY,
                  "gl": "id", "hl": "id", **params_extra}
        data = GoogleSearch(params).get_dict()
        _record_result(failure_counters, success=True)

        if engine == "google_shopping":
            items = data.get("shopping_results", [])
            return [
                {"source": it.get("source", "") or "Lainnya",
                 "title":  it.get("title", ""),
                 "price_text": _parse_price_idr(it.get("price", "")),
                 "product_url": it.get("product_link") or it.get("link", ""),
                 "image_url":  it.get("thumbnail", "")}
                for it in items
            ]
        if engine == "tokopedia":
            items = data.get("organic_results", [])
            return [
                {"source": "Tokopedia",
                 "title":  it.get("name", it.get("title", "")),
                 "price_text": _parse_price_idr(str(
                     it.get("price", {}).get("displayed", "")
                     if isinstance(it.get("price"), dict) else it.get("price", "")
                 )),
                 "product_url": it.get("url", it.get("link", "")),
                 "image_url":  it.get("image", "")}
                for it in items
            ]
        return []
    except Exception as e:
        print(f"  [warn] engine={engine} gagal: {e}")
        _record_result(failure_counters, success=False)
        return []

def _serpapi_google_shopping(query: str, failure_counters: dict,
                              limit: int = 10) -> list:
    if not SERPAPI_KEY or _circuit_open(failure_counters):
        return []
    for gl, q in [("id", query), ("us", f"{query} Indonesia beli online")]:
        time.sleep(random.uniform(*REQUEST_JITTER_RANGE_S))
        try:
            from serpapi import GoogleSearch
            data = GoogleSearch({
                "engine": "google_shopping",
                "q": q, "api_key": SERPAPI_KEY,
                "gl": gl, "hl": "id", "num": limit,
            }).get_dict()
            err = data.get("error")
            if err:
                print(f"  [warn] SerpApi error (gl={gl}): {err}")
                _record_result(failure_counters, success=False)
                continue
            _record_result(failure_counters, success=True)
            items = data.get("shopping_results", [])
            if items:
                results = []
                for it in items:
                    if not it.get("title"):
                        continue
                    link = it.get("product_link") or it.get("link", "")
                    if not link or "google.com" in link or not link.startswith("http"):
                        from urllib.parse import quote_plus
                        title_kw = quote_plus(it.get("title", query))
                        link = f"https://www.tokopedia.com/search?q={title_kw}&navsource=home&sort=6"
                    rating = it.get("rating")
                    try:
                        rating = float(rating) if rating else None
                    except Exception:
                        rating = None
                    reviews_count = it.get("reviews")
                    try:
                        reviews_count = int(reviews_count) if reviews_count else 0
                    except Exception:
                        reviews_count = 0
                    results.append({
                        "source":      it.get("source", "Google Shopping"),
                        "title":       it.get("title", ""),
                        "price_text":  _parse_price_idr(it.get("price", "")),
                        "product_url": link,
                        "image_url":   it.get("thumbnail", ""),
                        "rating":      rating,
                        "reviews_count": reviews_count,
                        "verified":    True,
                    })
                results.sort(key=lambda x: (x["rating"] is None, -(x["rating"] or 0)))
                return results
        except Exception as e:
            print(f"  [warn] SerpApi gl={gl} gagal: {e}")
            _record_result(failure_counters, success=False)
    return []

def _llm_product_catalog(category: str, room_type: str,
                          budget: int, style: str = "") -> list:
    from urllib.parse import quote_plus
    style_hint = f", gaya {style}" if style else ""
    prompt = f"""Kamu adalah asisten belanja Indonesia ahli furnitur dan dekorasi.
Kategori: "{category}" untuk {room_type}{style_hint}.
Budget per item: Rp{budget:,}.

Sebutkan 5 produk NYATA dengan nama merek/seri SPESIFIK yang populer dan banyak dijual di Indonesia.
Prioritaskan produk yang terkenal dan mudah ditemukan (contoh: IKEA, Olympic, Modena, Philips, Oxone, dll).
Harga harus dalam rentang Rp{int(budget*0.4):,}–Rp{int(budget*1.2):,}.
Format setiap baris: nama produk lengkap dengan merek dan seri | harga angka saja | Tokopedia/Shopee | lebar x dalam x tinggi (cm)
Contoh: Kasur Comforta Spring Bed Royal 120x200 | 2500000 | Tokopedia | 120x200x25
Contoh: Kompor Gas Rinnai RI-302S 2 Tungku | 450000 | Tokopedia | 57x37x13
Contoh: Meja Kerja IKEA Micke 73x50 Putih | 1200000 | Shopee | 73x50x75

Jawab HANYA 5 baris produk, TANPA komentar atau penjelasan lain."""
    try:
        raw = None
        for _m in GROQ_MODEL_CANDIDATES:
            try:
                resp = groq_client.chat.completions.create(
                    model=_m,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.2, max_tokens=400,
                )
                raw = resp.choices[0].message.content.strip()
                break
            except Exception:
                continue
        if not raw:
            return []
        results = []
        for line in raw.split("\n"):
            line = line.strip()
            if not line or "|" not in line:
                continue
            parts = [p.strip() for p in line.split("|")]
            if len(parts) < 2:
                continue
            name = parts[0].lstrip("0123456789.-) ").strip()
            if not name or len(name) < 3:
                continue
            raw_price = parts[1].strip() if len(parts) > 1 else str(budget)
            price_num = int(re.sub(r"[^0-9]", "", raw_price) or str(budget))
            if price_num <= 0:
                price_num = budget
            price_str = f"Rp{price_num:,}"
            src_mkt = parts[2].strip() if len(parts) > 2 else "Tokopedia"
            dims = parts[3].strip() if len(parts) > 3 else ""
            dm = re.findall(r'(\d+)[xX×](\d+)(?:[xX×](\d+))?', dims)
            w_cm = int(dm[0][0]) if dm else 60
            d_cm = int(dm[0][1]) if dm else 40
            h_cm = int(dm[0][2]) if dm and dm[0][2] else 60

            search_kw = quote_plus(name)
            if "shopee" in src_mkt.lower():
                product_url = f"https://shopee.co.id/search?keyword={search_kw}&sortBy=rating"
                platform_label = "Shopee"
            elif "bukalapak" in src_mkt.lower():
                product_url = f"https://www.bukalapak.com/products?search[keywords]={search_kw}&search[sort_by]=rating"
                platform_label = "Bukalapak"
            else:
                product_url = f"https://www.tokopedia.com/search?q={search_kw}&navsource=home&sort=6"
                platform_label = "Tokopedia"

            results.append({
                "source":        platform_label,
                "title":         name,
                "price_text":    price_str,
                "product_url":   product_url,
                "image_url":     "",
                "_dims_hint":    {"width_cm": w_cm, "depth_cm": d_cm, "height_cm": h_cm},
                "rating":        None,
                "reviews_count": 0,
                "verified":      True,
            })
        print(f"    [LLM catalog → Tokopedia/Shopee sort rating] {len(results)} produk untuk '{category}'")
        return results
    except Exception as e:
        print(f"  [warn] LLM catalog gagal: {e}")
        return []

def _tokopedia_search(query: str, limit: int = 10, min_rating: float = 4.0) -> list:
    from urllib.parse import quote_plus
    import requests as _req

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8",
        "Referer": f"https://www.tokopedia.com/search?q={quote_plus(query)}",
        "X-Source": "tokopedia-lite",
        "X-Version": "1.0.0",
    }

    ace_url = (
        "https://ace.tokopedia.com/search/product/v3"
        f"?q={quote_plus(query)}"
        f"&page=1&rows={limit}"
        f"&sort=5"
        f"&rt=4,5"
        f"&src=search"
        f"&device=desktop"
    )

    results = []
    try:
        resp = _req.get(ace_url, headers=headers, timeout=12)
        if resp.status_code == 200:
            data = resp.json()
            products = (
                data.get("data", {}).get("products", [])
                or data.get("products", [])
                or []
            )
            for p in products:
                name  = p.get("name", "")
                if not name:
                    continue
                price_int = p.get("price", {})
                if isinstance(price_int, dict):
                    price_int = price_int.get("value", 0) or price_int.get("text", 0)
                try:
                    price_int = int(str(price_int).replace(".", "").replace(",", "").replace("Rp","").strip())
                except Exception:
                    price_int = 0
                rating    = p.get("ratingAverage") or p.get("rating", None)
                try:
                    rating = float(rating) if rating else None
                except Exception:
                    rating = None
                reviews   = p.get("reviewCount") or p.get("review_count", 0)
                try:
                    reviews = int(reviews) if reviews else 0
                except Exception:
                    reviews = 0
                url_prod  = p.get("url", "")
                if url_prod and not url_prod.startswith("http"):
                    url_prod = "https://www.tokopedia.com" + url_prod
                shop_name = p.get("shop", {}).get("name", "") if isinstance(p.get("shop"), dict) else p.get("shopName", "")
                img       = p.get("imageUrl", "") or p.get("image_url", "")

                if not url_prod or "tokopedia.com" not in url_prod:
                    continue

                results.append({
                    "source":        f"Tokopedia — {shop_name}" if shop_name else "Tokopedia",
                    "title":         name,
                    "price_text":    f"Rp{price_int:,}" if price_int else "Rp-",
                    "product_url":   url_prod,
                    "image_url":     img,
                    "rating":        rating,
                    "reviews_count": reviews,
                    "verified":      True,
                })
    except Exception as _e:
        print(f"    [Tokopedia direct] gagal: {_e}")
        return []

    if not results:
        try:
            gql_url = "https://gql.tokopedia.com/graphql/SearchProductQueryV4"
            gql_headers = {**headers, "Content-Type": "application/json"}
            payload = [{
                "operationName": "SearchProductQueryV4",
                "variables": {"params": f"q={quote_plus(query)}&rows={limit}&sort=5&rt=4,5&st=product"},
                "query": (
                    "query SearchProductQueryV4($params:String){"
                    "ace_search_product_v4(params:$params){data{products{"
                    "id name url imageUrl500 price ratingAverage reviewCount "
                    "shop{name id}}}}}"),
            }]
            resp2 = _req.post(gql_url, json=payload, headers=gql_headers, timeout=12)
            if resp2.status_code == 200:
                d2 = resp2.json()
                prods2 = d2[0]["data"]["ace_search_product_v4"]["data"]["products"]
                for p2 in prods2:
                    url2 = p2.get("url","")
                    if not url2.startswith("http"):
                        url2 = "https://www.tokopedia.com" + url2
                    try:
                        price2 = int(str(p2.get("price","0")).replace("Rp","").replace(".","").replace(",","").strip())
                    except Exception:
                        price2 = 0
                    try:
                        r2v = float(p2.get("ratingAverage", 0)) if p2.get("ratingAverage") else None
                    except Exception:
                        r2v = None
                    results.append({
                        "source":        "Tokopedia — " + (p2.get("shop",{}).get("name","") or ""),
                        "title":         p2.get("name",""),
                        "price_text":    f"Rp{price2:,}" if price2 else "Rp-",
                        "product_url":   url2,
                        "image_url":     p2.get("imageUrl500",""),
                        "rating":        r2v,
                        "reviews_count": int(p2.get("reviewCount",0) or 0),
                        "verified":      True,
                    })
        except Exception as _e2:
            print(f"    [Tokopedia GQL] gagal: {_e2}")

    results.sort(key=lambda x: (
        -(x["rating"] or 0),
        -(x["reviews_count"] or 0),
    ))
    rated = [r for r in results if r["rating"] and r["rating"] >= min_rating]
    final = rated if rated else results
    print(f"    [Tokopedia direct] {len(final)} produk (rating≥{min_rating}) untuk '{query}'")
    return final[:limit]

def search_google_shopping(query: str, failure_counters: Dict[str, int],
                            limit: int = SEARCH_RESULT_LIMIT, category: str = "") -> List[Dict[str, Any]]:
    print(f"    [search] '{category or query}' → '{query}'")
    res = _serpapi_google_shopping(query, failure_counters, limit)
    if res:
        print(f"    [ok] SerpApi: {len(res)} produk")
    return res

def _search_with_fallback(item: Dict[str, Any], state: RoomPlannerState,
                          failure_counters: Dict[str, int]) -> List[Dict[str, Any]]:
    cat       = item["category"]
    room_type = state.get("room_type", "")
    style     = state.get("style_preference", "")
    est_price = item.get("estimated_price", 0)
    base_q    = _effective_query(item, state)

    queries = [
        base_q,
        f"{cat} {room_type}".strip(),
        f"jual {cat} murah".strip(),
        cat,
    ]
    seen, uq = set(), []
    for q in queries:
        if q and q not in seen:
            seen.add(q); uq.append(q)

    if SERPAPI_KEY and not _circuit_open(failure_counters):
        for q in uq:
            res = search_google_shopping(q, failure_counters, SEARCH_RESULT_LIMIT, cat)
            if res:
                print(f"    [SerpApi ✓] {len(res)} produk untuk '{cat}'")
                return res
            if _circuit_open(failure_counters):
                print(f"    [SerpApi] circuit open, skip ke Tokopedia direct")
                break
    else:
        print(f"    [SerpApi] skip (key kosong/circuit open) → coba Tokopedia direct")

    for q in uq[:2]:
        tokped_res = _tokopedia_search(q, limit=SEARCH_RESULT_LIMIT, min_rating=4.0)
        if tokped_res:
            print(f"    [Tokopedia direct ✓] {len(tokped_res)} produk rating≥4 untuk '{cat}'")
            return tokped_res

    print(f"    [LLM catalog] '{cat}' (semua search gagal)")
    return _llm_product_catalog(cat, room_type, est_price, style)

def _run_batch(items: List[Dict[str, Any]], state: RoomPlannerState,
               failure_counters: Dict[str, int]) -> Dict[str, List[Dict[str, Any]]]:
    results: Dict[str, List[Dict[str, Any]]] = {}
    if not items:
        return results
    with ThreadPoolExecutor(max_workers=min(SEARCH_MAX_WORKERS, len(items))) as pool:
        futures = {
            pool.submit(_search_with_fallback, item, state, failure_counters): item["category"]
            for item in items
        }
        for fut in as_completed(futures):
            results[futures[fut]] = fut.result()
    return results

def _items_to_search(state: RoomPlannerState) -> List[Dict[str, Any]]:
    all_items = state.get("recommended_items", [])
    retry_mods = state.get("retry_modifiers", {})
    if retry_mods:
        return [it for it in all_items if it["category"] in retry_mods]
    return all_items

def _split_batches(items: List[Dict[str, Any]]):
    mid = (len(items) + 1) // 2
    return items[:mid], items[mid:]

def search_batch_a_node(state: RoomPlannerState) -> Dict[str, Any]:
    print("\n[Product Search Agent A] mencari batch pertama kategori via Google Shopping (paralel dengan Agent B)...")
    targets = _items_to_search(state)
    batch_a, _ = _split_batches(targets)
    failure_counters: Dict[str, int] = {}
    results = dict(state.get("search_results_batch_a", {}))
    results.update(_run_batch(batch_a, state, failure_counters))
    return {"search_results_batch_a": results}

def search_batch_b_node(state: RoomPlannerState) -> Dict[str, Any]:
    print("[Product Search Agent B] mencari batch kedua kategori via Google Shopping (paralel dengan Agent A)...")
    targets = _items_to_search(state)
    _, batch_b = _split_batches(targets)
    failure_counters: Dict[str, int] = {}
    results = dict(state.get("search_results_batch_b", {}))
    results.update(_run_batch(batch_b, state, failure_counters))
    return {"search_results_batch_b": results}

## 8. Node 3c Search Merge Agent

In [ ]:
DIM_REGEX = re.compile(r"(\d{2,3})\s*[xX]\s*(\d{2,3})(?:\s*[xX]\s*(\d{2,3}))?\s*cm")

def extract_dims_from_title(title: str) -> Optional[Dict[str, int]]:
    m = DIM_REGEX.search(title or "")
    if not m:
        return None
    w, d, h = m.group(1), m.group(2), m.group(3)
    return {
        "width_cm": int(w),
        "depth_cm": int(d),
        "height_cm": int(h) if h else None,
    }

def _label_platform(source_text: str) -> str:
    s = (source_text or "").lower()
    if "tokopedia" in s:
        return "Tokopedia"
    if "shopee" in s:
        return "Shopee"
    if "bukalapak" in s:
        return "Bukalapak"
    if "blibli" in s:
        return "Blibli"
    if "lazada" in s:
        return "Lazada"
    cleaned = (source_text or "Toko Online").strip()
    return cleaned if cleaned else "Toko Online"

def search_merge_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Search Merge Agent] menggabungkan hasil Search Agent A + B, memberi label marketplace...")
    raw: Dict[str, List[Dict[str, Any]]] = {}
    raw.update(state.get("search_results_batch_a", {}))
    raw.update(state.get("search_results_batch_b", {}))

    merged = dict(state.get("search_results", {}))
    item_by_cat = {it["category"]: it for it in state["recommended_items"]}

    for cat, candidates in raw.items():
        suggested = item_by_cat.get(cat, {})
        for c in candidates:
            c["platform"] = _label_platform(c.pop("source", ""))
            if c.get("_dims_hint"):
                dims = c.pop("_dims_hint")
                c["dims_source"] = "llm_catalog"
            else:
                dims = extract_dims_from_title(c["title"])
                if not dims:
                    dims = {
                        "width_cm": suggested.get("suggested_width_cm"),
                        "depth_cm": suggested.get("suggested_depth_cm"),
                        "height_cm": suggested.get("suggested_height_cm"),
                    }
                    c["dims_source"] = "estimasi_llm"
                else:
                    c["dims_source"] = "judul_produk"
            c["dims_cm"] = dims
        merged[cat] = candidates
        n_tokped = sum(1 for c in candidates if c["platform"] == "Tokopedia")
        n_shopee = sum(1 for c in candidates if c["platform"] == "Shopee")
        n_other = len(candidates) - n_tokped - n_shopee
        print(f"  {cat}: {len(candidates)} kandidat (Tokopedia: {n_tokped}, Shopee: {n_shopee}, lainnya: {n_other})")

    if merged and all(len(v) == 0 for v in merged.values()):
        print("  [warn] SEMUA kategori nihil kandidat -- kemungkinan kuota SerpApi habis, "
              "API key salah, atau rate limit. (Bukan lagi bug filter site:, itu sudah diperbaiki.)")

    state["search_results"] = merged
    return state

## 9. Node 4 Evaluation Agent

## 9b. Node 4b Review Agent

In [ ]:
MIN_RATING = 4.0
MIN_REVIEWS = 10

def _get_rating_from_serpapi(query: str) -> dict:
    if not SERPAPI_KEY:
        return {"rating": None, "review_count": 0, "source": "serpapi_unavailable"}
    try:
        from serpapi import GoogleSearch
        time.sleep(random.uniform(0.1, 0.3))
        params = {"engine": "google_shopping", "q": query, "api_key": SERPAPI_KEY, "gl": "id", "hl": "id", "num": 5}
        results = GoogleSearch(params).get_dict()
        err = results.get("error")
        if err:
            return {"rating": None, "review_count": 0, "source": f"serpapi_error: {err}"}
        items = results.get("shopping_results", [])
        ratings = [float(it["rating"]) for it in items if it.get("rating")]
        reviews = [int(it.get("reviews", 0)) for it in items if it.get("reviews")]
        if ratings:
            return {
                "rating": round(sum(ratings) / len(ratings), 2),
                "review_count": max(reviews) if reviews else 0,
                "source": "google_shopping",
            }
        return {"rating": None, "review_count": 0, "source": "no_data"}
    except Exception as e:
        return {"rating": None, "review_count": 0, "source": f"error: {e}"}

def _heuristic_rating(category: str, platform: str, price: int) -> dict:
    return {"rating": None, "review_count": 0, "source": "heuristic_no_data"}

def review_agent_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Review Agent] mengecek rating & ulasan untuk setiap produk terpilih...")
    review_results = {}
    items_to_check = state.get("selected_items", [])
    if not items_to_check:
        state["review_results"] = {}
        return state

    def _check_one(sel):
        cat = sel["category"]
        product = sel["product"]
        platform = product.get("platform", "")
        price = parse_price(product.get("price_text", "0"))

        if product.get("verified") and product.get("rating") is not None:
            rating = product["rating"]
            rev_cnt = product.get("reviews_count", 0) or 0
            if rating < MIN_RATING:
                flag = f"⚠️ Rating rendah ({rating}) — pertimbangkan alternatif"
            elif rev_cnt < MIN_REVIEWS:
                flag = f"ℹ️ Review sedikit ({rev_cnt}) — belum banyak pembeli"
            else:
                flag = f"✅ Rating {rating} ({rev_cnt} ulasan)"
            return cat, {"rating": rating, "review_count": rev_cnt,
                         "source": "google_shopping_original", "flag": flag}

        if product.get("verified") is True and product.get("rating") is None:
            return cat, {"rating": None, "review_count": 0, "source": "llm_catalog_gshop",
                         "flag": "⭐ Klik link → masuk ke Tokopedia/Shopee, produk terurut rating tertinggi"}

        if product.get("verified") is False:
            return cat, {"rating": None, "review_count": 0, "source": "llm_catalog_unverified",
                         "flag": "ℹ️ Klik link → cari & bandingkan langsung di toko"}

        query = f"{cat} {product.get('title', '')}"[:60]
        rv = _get_rating_from_serpapi(query)
        if rv["rating"] is None:
            rv = _get_rating_from_serpapi(cat)
        if rv["rating"] is None:
            rv = _heuristic_rating(cat, platform, price)

        rating = rv.get("rating")
        rev_cnt = rv.get("review_count", 0)
        if rating and rating < MIN_RATING:
            flag = f"⚠️ Rating rendah ({rating}) — pertimbangkan alternatif"
        elif isinstance(rev_cnt, int) and rev_cnt < MIN_REVIEWS:
            flag = f"ℹ️ Review sedikit ({rev_cnt}) — belum banyak pembeli"
        else:
            flag = f"✅ Rating {rating} ({rev_cnt} ulasan)"
        return cat, {**rv, "flag": flag}

    with ThreadPoolExecutor(max_workers=min(4, len(items_to_check))) as pool:
        futures = {pool.submit(_check_one, sel): sel["category"] for sel in items_to_check}
        for fut in as_completed(futures):
            cat, result = fut.result()
            review_results[cat] = result
            print(f"  {cat}: {result['flag']}")

    state["review_results"] = review_results
    return state

print("Review Agent siap.")

## 9c. Node 4c Tukang Finder Agent

In [ ]:
MAX_TUKANG_PER_JENIS = 3

import functools

@functools.lru_cache(maxsize=64)
def _llm_tukang_query(jenis_tukang: str, room_type: str, item_name: str) -> str:
    _rt_clean   = room_type.strip()[:20]
    _item_clean = item_name.strip()[:20]
    _jenis_clean= jenis_tukang.replace("/", " atau ").strip()[:30]
    prompt = f"""Bantu cari tukang di Google Maps Indonesia.
Tukang: "{_jenis_clean}", item: "{_item_clean}", ruangan: "{_rt_clean}".
Tulis query 2-4 kata untuk Google Maps. Contoh: "tukang kayu custom solo" atau "jasa pasang keramik bandung".
Jawab HANYA querynya, tanpa tanda kutip, tanpa penjelasan."""
    try:
        raw = None
        for _model in GROQ_MODEL_CANDIDATES:
            try:
                resp = groq_client.chat.completions.create(
                    model=_model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.2, max_tokens=30,
                )
                raw = resp.choices[0].message.content.strip().strip('"').strip("'")
                raw = raw.split("\n")[0].strip()[:40]
                _words = set(raw.lower().split())
                _expect = set(jenis_tukang.lower().replace("/", " ").split()
                              + item_name.lower().split()
                              + ["tukang", "jasa", "pasang", "custom", "kayu",
                                 "keramik", "gypsum", "cat", "pipa", "listrik"])
                if raw and _words & _expect:
                    return raw
            except Exception:
                continue
    except Exception:
        pass
    base = jenis_tukang.replace("/", " ").replace("+", "").strip()
    return f"{base} {room_type}".strip()

def _get_tukang_query(jenis_tukang: str, room_type: str = "", item_name: str = "") -> str:
    return _llm_tukang_query(
        jenis_tukang=jenis_tukang,
        room_type=room_type,
        item_name=item_name,
    )

def _search_tukang_serpapi(query: str, location: str) -> list:
    if not SERPAPI_KEY:
        return []
    try:
        from serpapi import GoogleSearch
        params = {
            "engine":   "google_maps",
            "q":        f"{query} {location}",
            "api_key":  SERPAPI_KEY,
            "hl":       "id",
            "gl":       "id",
            "type":     "search",
        }
        results = GoogleSearch(params).get_dict()
        places  = results.get("local_results", [])
        out = []
        for p in places[:MAX_TUKANG_PER_JENIS]:
            out.append({
                "nama":       p.get("title", "-"),
                "alamat":     p.get("address", "-"),
                "rating":     p.get("rating"),
                "ulasan":     p.get("reviews"),
                "jarak":      p.get("distance", "-"),
                "telepon":    p.get("phone", "-"),
                "maps_url":   p.get("place_id_search") or p.get("link", "-"),
                "tipe":       query,
            })
        return out
    except Exception as e:
        return [{"nama": f"Gagal ({e})", "alamat": "-", "rating": None,
                 "ulasan": None, "jarak": "-", "telepon": "-",
                 "maps_url": "-", "tipe": query}]

def _heuristic_tukang(query: str, location: str) -> list:
    maps_search = f"https://www.google.com/maps/search/{query.replace(' ', '+')}+{location.replace(' ', '+')}"
    return [{
        "nama":     f"Cari '{query}' di Google Maps",
        "alamat":   f"Area: {location}",
        "rating":   None,
        "ulasan":   None,
        "jarak":    "-",
        "telepon":  "-",
        "maps_url": maps_search,
        "tipe":     query,
    }]

TUKANG_FURNITUR_QUERIES = [
    ("tukang kayu/meubel", "tukang kayu furniture custom"),
    ("jasa interior custom", "jasa interior furniture custom"),
]

def tukang_finder_node(state: RoomPlannerState) -> RoomPlannerState:
    custom_items = state.get("custom_items", [])
    location     = state.get("user_location", "").strip()
    room_type    = state.get("room_type", "")

    print(f"\n[Tukang Finder Agent] Mencari tukang di '{location or 'lokasi tidak diisi'}'...")

    tukang_results = {}

    def _fetch(jenis, query):
        results = _search_tukang_serpapi(query, location) if location else []
        if not results:
            results = _heuristic_tukang(query, location or "area kamu")
        return jenis, results

    if custom_items:
        jenis_set = {}
        for ci in custom_items:
            raw_jenis = ci.get("tukang", "tukang umum")
            query = _get_tukang_query(raw_jenis, room_type=room_type, item_name="")
            jenis_set[raw_jenis] = query

        with ThreadPoolExecutor(max_workers=min(4, len(jenis_set))) as pool:
            futures = {pool.submit(_fetch, j, q): j for j, q in jenis_set.items()}
            for fut in as_completed(futures):
                jenis, results = fut.result()
                tukang_results[jenis] = results
                found = len([r for r in results
                             if "Gagal" not in r["nama"] and "Cari" not in r["nama"]])
                print(f"  [custom item] {jenis}: {found} tukang ditemukan")
    else:
        print("  Tidak ada item custom — langsung ke tukang furnitur referensi.")

    print(f"  [furnitur custom] Mencari tukang kayu/meubel & jasa interior di '{location or 'area kamu'}'...")
    fur_results = {}
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = {pool.submit(_fetch, j, q): j for j, q in TUKANG_FURNITUR_QUERIES}
        for fut in as_completed(futures):
            jenis, results = fut.result()
            fur_results[jenis] = results
            found = len([r for r in results
                         if "Gagal" not in r["nama"] and "Cari" not in r["nama"]])
            print(f"  [furnitur custom] {jenis}: {found} tukang ditemukan")

    tukang_results["__furnitur_custom__"] = fur_results

    state["tukang_results"] = tukang_results
    return state

print("Tukang Finder Agent siap.")

In [ ]:
def _blocks_window(dims_cm: dict, layout_plan: dict, placement: str) -> bool:
    if placement != "dinding":
        return False
    h = dims_cm.get("height_cm", 0) or 0
    if h <= 100:
        return False
    return len(layout_plan.get("windows", [])) > 0

def fits_in_room(dims_cm: Dict[str, Any], layout_plan: Dict[str, Any]) -> bool:
    w = dims_cm.get("width_cm") or 0
    d = dims_cm.get("depth_cm") or 0
    if w <= 0 or d <= 0:
        return False
    w_m, d_m = w / 100, d / 100
    L, W = layout_plan["length_m"], layout_plan["width_m"]
    fits_normal = w_m <= L and (d_m + layout_plan["walkway_clearance_m"]) <= W
    fits_rotated = d_m <= L and (w_m + layout_plan["walkway_clearance_m"]) <= W
    return fits_normal or fits_rotated

def parse_price(price_text: str) -> int:
    digits = re.sub(r"[^\d]", "", price_text or "")
    return int(digits) if digits else 0

def _manual_search_url(category: str, room_type: str, prefer_platform: str = "tokopedia") -> str:
    from urllib.parse import quote_plus
    query = f"{category} {room_type}".strip()
    kw = quote_plus(query)
    if prefer_platform == "shopee":
        return f"https://shopee.co.id/search?keyword={kw}&sortBy=rating"
    return f"https://www.tokopedia.com/search?q={kw}&navsource=home&sort=6"

def _is_price_plausible(price: int, estimated_price) -> bool:
    if price <= 0 or not estimated_price or estimated_price <= 0:
        return True
    ratio = price / estimated_price
    return 0.25 <= ratio <= 4.0

import functools

@functools.lru_cache(maxsize=32)
def _llm_budget_weights(room_type: str, categories_tuple: tuple) -> dict:
    cats_str = ", ".join(f'"{c}"' for c in categories_tuple)
    prompt = f"""Kamu adalah konsultan anggaran interior Indonesia.
Ruangan: "{room_type}". Daftar kategori item: [{cats_str}].

Untuk setiap kategori, berikan BOBOT PRIORITAS BUDGET (float 0.3 – 4.0):
- Item UTAMA ruangan (misal kasur di kamar tidur, kitchen set di dapur) → bobot 2.5–4.0
- Item PENTING tapi bukan yang paling mahal (meja, lemari kecil) → bobot 1.5–2.5
- Item PENDUKUNG (kursi, rak kecil, nakas) → bobot 1.0–1.5
- Item DEKORATIF (cermin, lampu hias, hiasan, tanaman) → bobot 0.3–0.8

Jawab HANYA JSON object: {{"nama_kategori": bobot, ...}} tanpa teks lain."""
    try:
        raw = None
        for _model in GROQ_MODEL_CANDIDATES:
            try:
                resp = groq_client.chat.completions.create(
                    model=_model,
                    messages=[
                        {"role": "system", "content": "Jawab HANYA JSON valid."},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.1, max_tokens=300,
                )
                raw = resp.choices[0].message.content.strip()
                break
            except Exception:
                continue
        if raw is None:
            raise RuntimeError("Semua model gagal")
        import re as _re
        raw = _re.sub(r"```(?:json)?|```", "", raw).strip()
        result = json.loads(raw)
        validated = {}
        for cat, w in result.items():
            try:
                wf = float(w)
                validated[cat] = max(0.3, min(4.5, wf))
            except Exception:
                pass
        return validated
    except Exception as e:
        print(f"  [budget weight LLM gagal: {e} — pakai bobot default 1.0]")
        return {}

_budget_weight_cache: dict = {}

def _cat_weight(category: str, room_type: str = "", all_categories: tuple = ()) -> float:
    global _budget_weight_cache
    cache_key = (room_type, all_categories)
    if cache_key not in _budget_weight_cache:
        if all_categories:
            _budget_weight_cache[cache_key] = _llm_budget_weights(room_type, all_categories)
        else:
            _budget_weight_cache[cache_key] = {}
    weights = _budget_weight_cache[cache_key]
    cat_lower = category.lower()
    for k, w in weights.items():
        if k.lower() in cat_lower or cat_lower in k.lower():
            return w
    return 1.0

def evaluation_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Evaluation Agent] mengecek ukuran + harga + kompatibilitas lantai + jendela...")
    lp            = state["layout_plan"]
    all_items     = state["recommended_items"]
    floor_number  = state.get("floor_number", 1)
    budget_target_pct = state.get("budget_target_pct", 0.80)
    total_budget  = state["user_budget_total"]
    budget_target = total_budget * budget_target_pct

    selected        = []
    retry_modifiers = {}
    retry_count     = dict(state.get("retry_count", {}))
    total_spent     = 0

    def _closest(pool, target):
        def _sort_key(c):
            r = c.get("rating")
            try:
                r = float(r) if r else 0.0
            except Exception:
                r = 0.0
            price_diff = abs(parse_price(c.get("price_text", "0")) - target)
            return (-r, price_diff)
        return min(pool, key=_sort_key)

    room_type_state   = state.get("room_type", "")
    all_cats_tuple    = tuple(it["category"] for it in all_items)
    _cat_weight("__warmup__", room_type=room_type_state, all_categories=all_cats_tuple)

    for idx, item in enumerate(all_items):
        cat        = item["category"]
        candidates = state["search_results"].get(cat, [])

        if item.get("placement") == "langit-langit":
            if item.get("suggested_height_cm", 0) > 60:
                item["suggested_height_cm"] = 30
            if item.get("suggested_width_cm", 0) > 80:
                item["suggested_width_cm"] = 60
            if item.get("suggested_depth_cm", 0) > 80:
                item["suggested_depth_cm"] = 60
        floor_info = check_floor_compatibility(item, floor_number)
        if not floor_info["compatible"] and floor_number >= 2:
            _floor_note = floor_info['note']
            print(f"  ⚠️  '{cat}' bermasalah di lantai {floor_number}: {_floor_note}")
            item = dict(item)
            item["search_query"] = item.get("search_query", cat) + " knock down minimalis"

        remaining_items  = all_items[idx:]
        remaining_weight = sum(_cat_weight(it["category"], room_type_state, all_cats_tuple) for it in remaining_items) or 1
        remaining_budget = max(0, budget_target - total_spent)
        my_share         = (_cat_weight(cat, room_type_state, all_cats_tuple) / remaining_weight) * remaining_budget
        my_share_capped  = min(my_share, remaining_budget * 0.80)
        est_price        = item.get("estimated_price") or 0

        dims_ok = [c for c in candidates if fits_in_room(c["dims_cm"], lp)]

        if est_price > 0:
            ratio_cap = max(5.0, my_share_capped / est_price * 1.1)
        else:
            ratio_cap = 10.0

        def price_ok(price, _est=est_price, _cap=ratio_cap, _share=my_share_capped):
            if price <= 0: return True
            if _est <= 0:  return price <= _share
            return 0.15 <= (price / _est) <= _cap

        fitting = [c for c in dims_ok if price_ok(parse_price(c["price_text"]))]
        fitting_in_share = [c for c in fitting if parse_price(c["price_text"]) <= my_share_capped]

        _dbg_cat = cat
        print(f"  [debug] '{_dbg_cat}' (jatah Rp{my_share_capped:,.0f}, ratio_cap {ratio_cap:.1f}x): "
              f"{len(candidates)} -> {len(dims_ok)} muat -> "
              f"{len(fitting)} harga ok -> {len(fitting_in_share)} dalam jatah")

        if fitting_in_share:
            chosen = _closest(fitting_in_share, my_share_capped)
            total_spent += parse_price(chosen["price_text"])
            chosen["floor_compat"] = floor_info
            selected.append({"category": cat, "product": chosen,
                             "status": "✅ dari toko", "item_spec": item})
            continue

        if fitting:
            chosen = _closest(fitting, my_share_capped)
            total_spent += parse_price(chosen["price_text"])
            chosen["floor_compat"] = floor_info
            selected.append({"category": cat, "product": chosen,
                             "status": "✅ dari toko (sedikit di luar jatah)", "item_spec": item})
            continue

        retry_count[cat] = retry_count.get(cat, 0) + 1
        if retry_count[cat] <= 2:
            retry_modifiers[cat] = "simplify" if not candidates else "smaller_cheaper"
            print(f"  '{cat}' -> retry ({retry_count[cat]}/2)")
            continue

        if dims_ok:
            chosen = _closest(dims_ok, my_share_capped)
            chosen["floor_compat"] = floor_info
            total_spent += parse_price(chosen["price_text"])
            selected.append({"category": cat, "product": chosen,
                             "status": "✅ dari toko (di luar estimasi harga)", "item_spec": item})
            continue

        fallback_dims = {
            "width_cm":  item.get("suggested_width_cm"),
            "depth_cm":  item.get("suggested_depth_cm"),
            "height_cm": item.get("suggested_height_cm"),
        }
        if fits_in_room(fallback_dims, lp):
            capped_price = min(est_price, int(my_share_capped)) if est_price > 0 else int(my_share_capped * 0.5)
            remaining_now = budget_target - total_spent
            if capped_price > remaining_now * 1.05:
                print(f"  '{cat}' dilewati (estimasi Rp{capped_price:,} > sisa budget Rp{remaining_now:,})")
                continue
            _catalog_last_try = _llm_product_catalog(
                cat, state.get("room_type", ""), capped_price, state.get("style_preference", "")
            )
            _catalog_fitting = [c for c in _catalog_last_try
                                 if c.get("_dims_hint") and
                                 fits_in_room(c.get("_dims_hint", fallback_dims), lp)]
            if _catalog_fitting:
                chosen_fb = _catalog_fitting[0]
                chosen_fb["dims_cm"] = chosen_fb.pop("_dims_hint", fallback_dims)
                chosen_fb["dims_source"] = "llm_catalog"
                chosen_fb["floor_compat"] = floor_info
                chosen_fb["platform"] = _label_platform(chosen_fb.get("source", "Tokopedia"))
                actual_price = parse_price(chosen_fb.get("price_text", "0")) or capped_price
                if actual_price > (budget_target - total_spent) * 1.05:
                    actual_price = capped_price
                    chosen_fb["price_text"] = f"Rp{capped_price:,}"
                total_spent += actual_price
                selected.append({"category": cat, "product": chosen_fb,
                                 "status": "dari toko (LLM catalog)", "item_spec": item})
                print(f"  '{cat}': produk LLM catalog — {chosen_fb['title'][:50]}")
            else:
                _prefer_plat = "shopee" if (idx % 2 == 1) else "tokopedia"
                fallback_product = {
                    "platform":    "Tokopedia" if _prefer_plat == "tokopedia" else "Shopee",
                    "title":       f"{cat} — klik untuk lihat pilihan terlaris",
                    "price_text":  f"Rp{capped_price:,}",
                    "product_url": _manual_search_url(cat, state.get("room_type", ""), _prefer_plat),
                    "dims_cm":     fallback_dims,
                    "dims_source": "estimasi_llm",
                    "floor_compat": floor_info,
                }
                total_spent += capped_price
                selected.append({"category": cat, "product": fallback_product,
                                 "status": "cari di toko (klik link)", "item_spec": item})
        else:
            print(f"  '{cat}' tidak muat setelah 2x retry — dilewati.")

    state["retry_count"]     = retry_count
    state["retry_modifiers"] = retry_modifiers
    state["_needs_retry"]    = len(retry_modifiers) > 0

    budget_used_pct = (total_spent / total_budget * 100) if total_budget > 0 else 0
    shortfall_pct   = budget_target_pct * 100 - budget_used_pct

    if shortfall_pct > 8 and not state["_needs_retry"]:
        sisa_budget = int(budget_target - total_spent)
        selected_cats = [s["category"] for s in selected]
        room_type_tu  = state.get("room_type", "ruangan")
        style_tu      = state.get("style_preference") or "netral"
        lp_tu         = state["layout_plan"]
        print(f"  ⚠️  Budget baru {budget_used_pct:.1f}% (target {budget_target_pct*100:.0f}%).")

        topup_prompt = f"""Kamu adalah interior designer Indonesia.
Ruangan: "{room_type_tu}", ukuran {lp_tu['length_m']}x{lp_tu['width_m']}m, gaya: {style_tu}.
Budget sisa yang belum terpakai: Rp{sisa_budget:,}

Item yang SUDAH ada di ruangan: {', '.join(selected_cats)}

Usulkan 2-4 item TAMBAHAN untuk melengkapi ruangan dan menghabiskan sisa budget Rp{sisa_budget:,}.
Jangan duplikat item yang sudah ada. Pilih item yang nilainya sepadan dengan sisa budget.
Barang nyata yang dijual di Tokopedia/Shopee.

Jawab HANYA JSON list dengan field: "category","search_query","estimated_price",
"suggested_width_cm","suggested_depth_cm","suggested_height_cm","placement","priority","reasoning"
placement: "dinding"/"lantai"/"langit-langit". priority: 10-20 (tambahan, bukan utama)."""
        try:
            raw_topup = ask_llm_json(topup_prompt)
            topup_items = _sanitize_items(raw_topup, max_count=4, default_placement="lantai")
            topup_items = [it for it in topup_items
                           if not any(_is_probable_duplicate(it["category"], sc,
                                       ignore_words=_normalize_words(room_type_tu))
                                      for sc in selected_cats)]
            if topup_items:
                print(f"  💡 LLM mengusulkan {len(topup_items)} barang tambahan buat menghabiskan sisa budget Rp{sisa_budget:,}:")
                for it in topup_items:
                    print(f"     - {it['category']} (~Rp{it.get('estimated_price', 0):,})")
                _ans_tu = input("  Tambahkan barang-barang ini ke keranjang & ruangan? (ya/tidak): ").strip().lower()
                if _ans_tu not in ("ya", "y", "yes"):
                    print("  (Dilewati sesuai pilihan kamu -- tidak ada barang tambahan yang ditambahkan/ditata.)")
                    topup_items = []
            if topup_items:
                print(f"  ✅ Menambahkan {len(topup_items)} item top-up:")
                for it in topup_items:
                    it["source_agent"] = "budget_topup"
                    topup_cat = it["category"]
                    topup_cands = state["search_results"].get(topup_cat, [])
                    topup_dims_ok = [c for c in topup_cands if fits_in_room(c["dims_cm"], lp)]
                    est = it.get("estimated_price", 0)
                    if topup_dims_ok:
                        chosen = min(topup_dims_ok, key=lambda c: abs(parse_price(c["price_text"]) - sisa_budget // max(1, len(topup_items))))
                        total_spent += parse_price(chosen["price_text"])
                        fc = check_floor_compatibility(it, floor_number)
                        chosen["floor_compat"] = fc
                        selected.append({"category": topup_cat, "product": chosen,
                                         "status": "top-up budget", "item_spec": it})
                        print(f"    + {topup_cat}: {chosen['title'][:40]} — {chosen['price_text']}")
                    elif est > 0 and est <= sisa_budget:
                        _topup_catalog = _llm_product_catalog(topup_cat, room_type_tu, est, style_tu)
                        _topup_dims = {"width_cm": it.get("suggested_width_cm", 60),
                                       "depth_cm": it.get("suggested_depth_cm", 40),
                                       "height_cm": it.get("suggested_height_cm", 60)}
                        if _topup_catalog:
                            _tc = _topup_catalog[0]
                            _tc["dims_cm"] = _tc.pop("_dims_hint", _topup_dims)
                            _tc["dims_source"] = "llm_catalog"
                            _tc["floor_compat"] = check_floor_compatibility(it, floor_number)
                            _tc["platform"] = _label_platform(_tc.get("source", "Tokopedia"))
                            total_spent += parse_price(_tc.get("price_text", f"Rp{est:,}")) or est
                            selected.append({"category": topup_cat, "product": _tc,
                                             "status": "top-up dari toko", "item_spec": it})
                            print(f"    + {topup_cat}: {_tc['title'][:40]} — {_tc['price_text']}")
                        else:
                            _prefer_tu = "shopee" if len(selected) % 2 == 0 else "tokopedia"
                            fallback_p = {
                                "platform": "Tokopedia" if _prefer_tu == "tokopedia" else "Shopee",
                                "title": f"{topup_cat} — klik untuk lihat pilihan terlaris",
                                "price_text": f"Rp{est:,}",
                                "product_url": _manual_search_url(topup_cat, room_type_tu, _prefer_tu),
                                "dims_cm": _topup_dims,
                                "dims_source": "estimasi_llm",
                                "floor_compat": check_floor_compatibility(it, floor_number),
                            }
                            total_spent += est
                            selected.append({"category": topup_cat, "product": fallback_p,
                                             "status": "top-up (klik link)", "item_spec": it})
                            print(f"    + {topup_cat}: link toko Rp{est:,}")
                    selected_cats.append(topup_cat)
        except Exception as e:
            print(f"  (top-up LLM gagal: {e})")

    state["selected_items"]  = selected
    budget_used_pct = (total_spent / total_budget * 100) if total_budget > 0 else 0
    _bar = "█" * int(budget_used_pct / 5) + "░" * (20 - int(budget_used_pct / 5))
    print(f"\n  📊 Budget: Rp{total_spent:,} / Rp{total_budget:,} "
          f"({budget_used_pct:.1f}%) [{_bar}] target {budget_target_pct*100:.0f}%")
    return state

def route_after_evaluation(state: RoomPlannerState) -> str:
    return "retry" if state.get("_needs_retry") else "proceed"

## 11. Node 5 Placement Planner Agent

In [ ]:
def _rects_overlap(a, b) -> bool:
    return not (a["x"] + a["w"] <= b["x"] or b["x"] + b["w"] <= a["x"] or
                a["y"] + a["d"] <= b["y"] or b["y"] + b["d"] <= a["y"])

def placement_planner_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Placement Planner Agent] menyusun posisi tiap barang di dalam ruangan...")
    lp = state["layout_plan"]
    L, W = lp["length_m"], lp["width_m"]
    door = lp["door_zone"]

    ceiling_selected = [s for s in state["selected_items"] if s["item_spec"].get("placement") == "langit-langit"]
    floor_wall_selected = [s for s in state["selected_items"] if s["item_spec"].get("placement") != "langit-langit"]
    state["ceiling_items"] = [{"category": s["category"], "label": s["product"]["title"][:40]} for s in ceiling_selected]
    for c in state["ceiling_items"]:
        print(f"  '{c['category']}' ditandai sebagai barang langit-langit -- tidak digambar sebagai kotak lantai.")

    placed = []

    def collides_with_door(rect):
        return not (rect["x"] + rect["w"] <= door["x_min"] or door["x_max"] <= rect["x"] or
                    rect["y"] + rect["d"] <= door["y_min"] or door["y_max"] <= rect["y"])

    def find_spot(w_m, d_m, prefer_wall: bool):
        step = 0.2
        candidate_positions = []
        if prefer_wall:
            candidate_positions += [(x, 0) for x in _frange(0, L - w_m, step)]
            candidate_positions += [(x, W - d_m) for x in _frange(0, L - w_m, step)]
            candidate_positions += [(0, y) for y in _frange(0, W - d_m, step)]
            candidate_positions += [(L - w_m, y) for y in _frange(0, W - d_m, step)]
        candidate_positions += [(x, y) for x in _frange(0, L - w_m, step) for y in _frange(0, W - d_m, step)]

        for x, y in candidate_positions:
            rect = {"x": round(x, 2), "y": round(y, 2), "w": round(w_m, 2), "d": round(d_m, 2)}
            if collides_with_door(rect):
                continue
            if any(_rects_overlap(rect, p) for p in placed):
                continue
            return rect
        return None

    def _frange(a, b, step):
        vals = []
        x = a
        while x <= b + 1e-9:
            vals.append(round(x, 2))
            x += step
        return vals or [a]

    items_sorted = sorted(
        floor_wall_selected,
        key=lambda s: (s["item_spec"].get("priority", 99),
                        -(s["product"]["dims_cm"].get("width_cm") or 0) * (s["product"]["dims_cm"].get("depth_cm") or 0)),
    )

    _SOFT_KEYWORDS = {"karpet", "rug", "permadani", "matras", "tikar"}
    _MAX_SOFT_RATIO = 0.45

    for sel in items_sorted:
        cat  = sel["category"]
        dims = sel["product"]["dims_cm"]
        w_m = (dims.get("width_cm") or 60) / 100
        d_m = (dims.get("depth_cm") or 60) / 100
        prefer_wall = sel["item_spec"].get("placement") == "dinding"

        if any(k in cat.lower() for k in _SOFT_KEYWORDS):
            max_side = (L * W * _MAX_SOFT_RATIO) ** 0.5
            if w_m > max_side or d_m > max_side:
                scale = max_side / max(w_m, d_m)
                w_m = round(w_m * scale, 2)
                d_m = round(d_m * scale, 2)
                print(f"  [{cat}] ukuran karpet dikecilkan → {w_m}m x {d_m}m")

        if w_m > L or d_m > W:
            w_m, d_m = min(w_m, L), min(d_m, W)

        rect = find_spot(w_m, d_m, prefer_wall)
        if rect is None:
            print(f"  [warn] tidak menemukan slot kosong untuk '{sel['category']}', dilewati dari layout akhir.")
            continue

        rect.update({"category": sel["category"], "label": sel["product"]["title"][:28]})
        placed.append(rect)

    state["placement"] = placed
    print(f"  {len(placed)} dari {len(state['selected_items'])} barang berhasil ditempatkan di layout.")
    return state

## 12. Node 6 Floorplan Visualizer Agent

In [ ]:
def floorplan_visualizer_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Floorplan Visualizer Agent] menggambar 2D floor plan skala nyata (enhanced)...")
    lp = state["layout_plan"]
    L, W = lp["length_m"], lp["width_m"]
    door = lp["door_zone"]
    floor_number = state.get("floor_number", 1)
    windows = lp.get("windows", [])

    fig, ax = plt.subplots(figsize=(10, 10 * W / L if L else 10))
    ax.set_xlim(-0.5, L + 0.5)
    ax.set_ylim(-0.5, W + 0.5)
    ax.set_aspect("equal")

    import matplotlib.patches as patches
    room_bg = patches.Rectangle((0, 0), L, W, facecolor="#F5F0EB", edgecolor="black", linewidth=3)
    ax.add_patch(room_bg)

    for xi in [x * 0.5 for x in range(int(L * 2) + 1)]:
        ax.axvline(xi, color="#E0D8D0", linewidth=0.4, zorder=1)
    for yi in [y * 0.5 for y in range(int(W * 2) + 1)]:
        ax.axhline(yi, color="#E0D8D0", linewidth=0.4, zorder=1)

    wall_t = 0.08
    for rx, ry, rw, rd in [
        (0, 0, L, wall_t), (0, W - wall_t, L, wall_t),
        (0, 0, wall_t, W), (L - wall_t, 0, wall_t, W)
    ]:
        ax.add_patch(patches.Rectangle((rx, ry), rw, rd, facecolor="#555", zorder=3))

    door_cx = (door["x_min"] + door["x_max"]) / 2
    door_w = door["x_max"] - door["x_min"]
    ax.add_patch(patches.Rectangle((door["x_min"], -0.02), door_w, wall_t + 0.02,
                                    facecolor="#F5F0EB", edgecolor="none", zorder=4))
    ax.add_patch(patches.Arc((door["x_min"], 0), door_w, door_w,
                              angle=0, theta1=0, theta2=90,
                              color="#8B4513", linewidth=2, zorder=5))
    ax.plot([door["x_min"], door["x_min"]], [0, door_w / 2], color="#8B4513", linewidth=2, zorder=5)
    ax.text(door_cx, -0.3, "[D] PINTU", ha="center", fontsize=8, color="#8B4513", fontweight="bold")

    for win in windows:
        wall = win.get("wall", "atas")
        c = win.get("center_m", L / 2)
        ww = win.get("width_m", 0.9)
        wh = 0.1
        if wall == "bawah":
            rx, ry = c - ww/2, 0
            rw, rd = ww, wh
        elif wall == "atas":
            rx, ry = c - ww/2, W - wh
            rw, rd = ww, wh
        elif wall == "kiri":
            rx, ry = 0, c - ww/2
            rw, rd = wh, ww
        else:
            rx, ry = L - wh, c - ww/2
            rw, rd = wh, ww

        ax.add_patch(patches.Rectangle((rx, ry), rw, rd, facecolor="#AEE4FF",
                                        edgecolor="#4A90D9", linewidth=2, zorder=6))
        mid_x = rx + rw / 2
        mid_y = ry + rd / 2
        if wall in ("bawah", "atas"):
            ax.plot([mid_x, mid_x], [ry, ry + rd], color="#4A90D9", linewidth=1, zorder=7)
        else:
            ax.plot([rx, rx + rw], [mid_y, mid_y], color="#4A90D9", linewidth=1, zorder=7)
        ax.text(mid_x, mid_y + (0.25 if wall == "atas" else -0.25),
                "[W]", ha="center", va="center", fontsize=10, zorder=8)

    ax.add_patch(patches.Rectangle(
        (door["x_min"] - 0.1, 0), door["x_max"] - door["x_min"] + 0.2, lp["walkway_clearance_m"],
        facecolor="#FFF3CD", edgecolor="#FFC107", linewidth=1, linestyle="--", alpha=0.5, zorder=2
    ))

    FURNITURE_ICONS = {
        "kasur": "BED", "tempat tidur": "BED", "sofa": "SFA", "meja": "KRS", "kursi": "KRS",
        "lemari": "CAB", "almari": "CAB", "rak": "RAK", "tv": "TV", "televisi": "TV",
        "wastafel": "SHW", "toilet": "WC", "bathtub": "TUB", "kulkas": "FRZ",
        "lampu": "LMP", "tanaman": "PLT", "karpet": "[]", "meja kerja": "PC",
    }

    palette = [
        "#FF8A80", "#EA80FC", "#82B1FF", "#69F0AE", "#FFD180",
        "#A7FFEB", "#CCFF90", "#84FFFF", "#B9F6CA", "#FFE57F",
        "#F8BBD0", "#D1C4E9", "#BBDEFB", "#B2EBF2", "#C8E6C9",
    ]

    for i, rect in enumerate(state["placement"]):
        color = palette[i % len(palette)]
        cat_lower = rect["category"].lower()
        icon = next((v for k, v in FURNITURE_ICONS.items() if k in cat_lower), "BOX")

        shadow = patches.Rectangle(
            (rect["x"] + 0.04, rect["y"] - 0.04), rect["w"], rect["d"],
            facecolor="#00000020", edgecolor="none", zorder=8
        )
        ax.add_patch(shadow)
        furn = patches.Rectangle(
            (rect["x"], rect["y"]), rect["w"], rect["d"],
            facecolor=color, edgecolor="#333", linewidth=1.5, alpha=0.95, zorder=9
        )
        ax.add_patch(furn)

        cx = rect["x"] + rect["w"] / 2
        cy = rect["y"] + rect["d"] / 2
        label = rect["category"]
        if len(label) > 14:
            mid = len(label) // 2
            space = label.rfind(" ", 0, mid + 2)
            if space > 0:
                label = label[:space] + "\n" + label[space+1:]
        ax.text(cx, cy + 0.05, f"{icon}\n{label}",
                ha="center", va="center", fontsize=7, fontweight="bold",
                zorder=10, multialignment="center",
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.6, edgecolor="none"))

        ax.text(rect["x"] + 0.03, rect["y"] + 0.03,
                f"{rect['w']*100:.0f}x{rect['d']*100:.0f}cm",
                fontsize=5.5, color="#555", zorder=10)

    ceiling_items = state.get("ceiling_items", [])
    if ceiling_items:
        names = ", ".join(f"LMP{c['category']}" for c in ceiling_items)
        ax.text(L / 2, W + 0.35, f"Langit-langit: {names}",
                ha="center", fontsize=8, color="#555",
                bbox=dict(boxstyle="round", facecolor="#FFFDE7", edgecolor="#FDD835", alpha=0.9))

    budget = state["user_budget_total"]
    spent = sum(
        parse_price(s["product"].get("price_text", "0"))
        for s in state.get("selected_items", [])
    )
    pct = spent / budget * 100 if budget else 0
    bar_color = "#4CAF50" if 65 <= pct <= 90 else ("#FF9800" if pct < 65 else "#F44336")
    budget_text = (f"Budget: Rp{budget:,}\n"
                   f"Terpakai: Rp{spent:,} ({pct:.1f}%)\n"
                   f"Sisa: Rp{budget-spent:,}")
    ax.text(L + 0.45, W - 0.1, budget_text, va="top", fontsize=7.5,
            bbox=dict(boxstyle="round", facecolor=bar_color, alpha=0.85, edgecolor="white"),
            color="white", fontweight="bold")

    floor_label = f"Lantai {floor_number}" if floor_number > 1 else "Lantai 1 (Ground)"
    title = (f"Floor Plan — {state['room_type'].title()} | {L}m × {W}m × {lp['height_m']}m | {floor_label}\n"
             f"{len(state['placement'])} item ditempatkan | {len(windows)} jendela | "
             f"Budget terpakai {pct:.1f}%")
    ax.set_title(title, fontsize=10, fontweight="bold", pad=12)
    ax.set_xlabel("meter →", fontsize=9)
    ax.set_ylabel("meter ↑", fontsize=9)

    from matplotlib.lines import Line2D
    scale_bar = Line2D([L - 1, L], [-0.4, -0.4], color="black", linewidth=2)
    ax.add_line(scale_bar)
    ax.text(L - 0.5, -0.45, "1 m", ha="center", fontsize=7)

    out_path = "final_floorplan.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    state["final_floorplan_path"] = out_path
    print(f"  Floor plan (enhanced) disimpan: {out_path}")
    return state

## 13b. Node 7 3D Interior Image Generator

In [ ]:
HF_MODELS = [
    "black-forest-labs/FLUX.1-schnell",
    "stabilityai/stable-diffusion-xl-base-1.0",
    "runwayml/stable-diffusion-v1-5",
]

def _build_interior_prompt(state: RoomPlannerState) -> str:
    room_type = state.get("room_type", "living room")
    style     = state.get("style_preference", "") or "modern minimalist"
    lp        = state.get("layout_plan", {})
    L, W, H   = lp.get("length_m", 4), lp.get("width_m", 3), lp.get("height_m", 3)
    selected  = state.get("selected_items", [])

    furniture_list = [
        f"{s.get('category','')} ({s.get('product',{}).get('title','')[:35]})"
        for s in selected if s.get("category")
    ]
    furniture_str = ", ".join(furniture_list[:10]) or "furniture"

    room_map = {
        "ruang tamu": "living room", "kamar tidur": "bedroom",
        "dapur": "kitchen", "kamar mandi": "bathroom",
        "ruang kerja": "home office", "ruang keluarga": "family room",
        "ruang makan": "dining room", "teras": "terrace",
    }
    room_en = room_map.get(room_type.lower(), room_type)

    prompt_req = f"""You are an interior design photography prompt expert.
Generate ONE detailed English prompt for an AI image generator to create a photorealistic 3D interior render.

Room: {room_en}, {L}m x {W}m x {H}m ceiling, style: {style}
Key furniture: {furniture_str}

Rules:
- Start with: "Photorealistic interior design render of a {room_en},"
- Include lighting, materials, color palette, camera angle (eye-level)
- Mention 3-5 key furniture pieces naturally
- End with: "professional interior photography, 8K, high detail, architectural visualization"
- Max 120 words, ONE paragraph, no line breaks

Output ONLY the prompt text."""

    try:
        for model in GROQ_MODEL_CANDIDATES:
            try:
                resp = groq_client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt_req}],
                    temperature=0.7, max_tokens=200,
                )
                raw = resp.choices[0].message.content.strip()
                for prefix in ["Prompt:", "Here is", "OUTPUT:", "```"]:
                    if raw.lower().startswith(prefix.lower()):
                        raw = raw[len(prefix):].strip().strip(":`")
                return raw
            except Exception:
                continue
    except Exception as e:
        print(f"  [warn] LLM prompt gagal: {e}")

    room_en_fb = room_map.get(state.get("room_type", "").lower(), "living room")
    style_fb   = state.get("style_preference", "") or "modern minimalist"
    return (
        f"Photorealistic interior design render of a {room_en_fb}, {style_fb} style, "
        f"warm natural lighting, wooden floors, white walls, tasteful furniture arrangement, "
        f"large window with natural daylight, clean and elegant atmosphere, "
        f"professional interior photography, 8K, high detail, architectural visualization"
    )

def _generate_image_hf(prompt: str, out_path: str = "interior_3d.png") -> str | None:
    if not HF_TOKEN:
        return None

    import requests as _req
    headers = {
        "Authorization": f"Bearer {HF_TOKEN}",
        "Content-Type": "application/json",
    }

    for model_id in HF_MODELS:
        url = f"https://api-inference.huggingface.co/models/{model_id}"
        payload = {
            "inputs": prompt,
            "parameters": {
                "num_inference_steps": 4 if "schnell" in model_id else 25,
                "guidance_scale":      0.0 if "schnell" in model_id else 7.5,
                "width": 1024, "height": 768,
            },
            "options": {"wait_for_model": True},
        }
        print(f"  Mencoba: {model_id.split('/')[-1]} ...")
        try:
            resp = _req.post(url, headers=headers, json=payload, timeout=120)
            if resp.status_code == 200 and resp.headers.get("content-type", "").startswith("image"):
                with open(out_path, "wb") as f_img:
                    f_img.write(resp.content)
                print(f"  ✅ Gambar tersimpan: {out_path} ({len(resp.content)//1024} KB)")
                return out_path
            elif resp.status_code == 503:
                print(f"  Model loading, coba berikutnya...")
                continue
            else:
                print(f"  Error {resp.status_code}: {resp.text[:150]}")
                continue
        except Exception as e:
            print(f"  Gagal: {e}")
            continue

    print("  Semua model gagal.")
    return None

def interior_image_generator_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Node 7 — 3D Interior Image Generator]")

    if not HF_TOKEN:
        print("  ⚠️  HF_TOKEN tidak ada → skip")
        print("  Daftar token gratis: https://huggingface.co/settings/tokens")
        state["generated_image_path"] = ""
        return state

    print("  Step 1/2: Generate prompt via Groq...")
    prompt = _build_interior_prompt(state)
    print(f"  Prompt: {prompt[:100]}...")

    print("  Step 2/2: Generate gambar via Hugging Face...")
    result = _generate_image_hf(prompt, "interior_3d.png")
    state["generated_image_path"] = result or ""
    return state

## 13. Bangun Graph

In [ ]:
def retry_dispatch_node(state: RoomPlannerState) -> RoomPlannerState:
    print("\n[Retry Dispatch] memicu ulang Search Agent A + B (paralel) untuk kategori yang belum cocok...")
    return state

graph = StateGraph(RoomPlannerState)

graph.add_node("layout_planner", layout_planner_node)
graph.add_node("core_furniture", core_furniture_node)
graph.add_node("theme_decor", theme_decor_node)
graph.add_node("recommendation_merge", recommendation_merge_node)
graph.add_node("item_selection", item_selection_node)
graph.add_node("search_batch_a", search_batch_a_node)
graph.add_node("search_batch_b", search_batch_b_node)
graph.add_node("search_merge", search_merge_node)
graph.add_node("evaluation", evaluation_node)
graph.add_node("retry_dispatch", retry_dispatch_node)
graph.add_node("placement_planner", placement_planner_node)
graph.add_node("review_agent", review_agent_node)
graph.add_node("tukang_finder", tukang_finder_node)
graph.add_node("floorplan_visualizer", floorplan_visualizer_node)
graph.add_node("interior_image_generator", interior_image_generator_node)
graph.set_entry_point("layout_planner")

graph.add_edge("layout_planner", "core_furniture")
graph.add_edge("layout_planner", "theme_decor")
graph.add_edge("core_furniture", "recommendation_merge")
graph.add_edge("theme_decor", "recommendation_merge")

graph.add_edge("recommendation_merge", "item_selection")

graph.add_edge("item_selection", "search_batch_a")
graph.add_edge("item_selection", "search_batch_b")
graph.add_edge("retry_dispatch", "search_batch_a")
graph.add_edge("retry_dispatch", "search_batch_b")
graph.add_edge("search_batch_a", "search_merge")
graph.add_edge("search_batch_b", "search_merge")

graph.add_edge("search_merge", "evaluation")

graph.add_conditional_edges(
    "evaluation",
    route_after_evaluation,
    {"retry": "retry_dispatch", "proceed": "placement_planner"},
)

graph.add_edge("placement_planner", "review_agent")
graph.add_edge("review_agent", "tukang_finder")
graph.add_edge("tukang_finder", "floorplan_visualizer")
graph.add_edge("floorplan_visualizer", "interior_image_generator")
graph.add_edge("interior_image_generator", END)

app = graph.compile()
print("Graph berhasil dibangun: 2 pasang agent paralel (rekomendasi & pencarian Google Shopping) + retry loop.")

## 14. Input dari User

In [ ]:
def _generate_custom_types_llm(room_type: str) -> dict:
    print(f"  (AI sedang menyusun daftar custom item untuk '{room_type}'...)")
    try:
        prompt = f"""Kamu adalah konsultan renovasi interior Indonesia.
Untuk ruangan "{room_type}", berikan 6-8 elemen yang HARUS dikerjakan tukang (bukan beli jadi).

WAJIB: Setiap item HARUS punya jenis tukang SPESIFIK dari daftar ini:
- tukang kayu/meubel  → untuk furniture kayu, kitchen set, lemari built-in, rak kayu
- tukang keramik      → untuk pasang keramik, backsplash, lantai
- tukang gypsum       → untuk plafon gypsum, partisi gypsum
- tukang cat/wallpaper→ untuk cat dinding, wallpaper, tekstur dinding
- tukang pipa         → untuk instalasi pipa air, wastafel tanam, shower
- tukang listrik      → untuk instalasi listrik, stop kontak, lampu tanam
- tukang besi         → untuk railing, tangga besi, rangka logam
- tukang kayu/aluminium → untuk kusen, pintu geser, partisi aluminium

JANGAN gunakan "tukang umum" — selalu pilih jenis tukang spesifik di atas.

Contoh untuk dapur: kitchen set → tukang kayu/meubel, backsplash keramik → tukang keramik, instalasi sink → tukang pipa
Contoh untuk kamar mandi: shower niche → tukang keramik, wastafel custom → tukang pipa, cermin tanam → tukang kayu/meubel

Balas HANYA JSON tanpa teks lain:
{{
  "nama item": {{"tukang": "<SPESIFIK dari daftar>", "material_utama": ["material1", "material2"]}},
  "nama item 2": {{"tukang": "<SPESIFIK dari daftar>", "material_utama": ["material1", "material2"]}}
}}

Buat untuk "{room_type}" — pilih yang paling relevan dan sering dibutuhkan."""

        raw = None
        for _model in GROQ_MODEL_CANDIDATES:
            try:
                response = groq_client.chat.completions.create(
                    model=_model,
                    messages=[
                        {"role": "system", "content": "Kamu selalu menjawab HANYA dengan JSON valid, tanpa teks lain, tanpa markdown code fence."},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.3,
                    max_tokens=700,
                )
                raw = response.choices[0].message.content.strip()
                break
            except Exception as _e:
                print(f"  (model {_model} gagal: {_e}, coba model berikutnya...)")
                continue
        if raw is None:
            raise RuntimeError("Semua model Groq gagal dipanggil.")
        raw = re.sub(r"```(?:json)?|```", "", raw).strip()
        generated = json.loads(raw)
        _tukang_map_fix = {
            "kayu": "tukang kayu/meubel", "meubel": "tukang kayu/meubel",
            "lemari": "tukang kayu/meubel", "rak": "tukang kayu/meubel",
            "kitchen": "tukang kayu/meubel", "kabinet": "tukang kayu/meubel",
            "keramik": "tukang keramik", "backsplash": "tukang keramik",
            "lantai": "tukang keramik", "mozaik": "tukang keramik",
            "gypsum": "tukang gypsum", "plafon": "tukang gypsum",
            "cat": "tukang cat/wallpaper", "wallpaper": "tukang cat/wallpaper",
            "pipa": "tukang pipa", "sink": "tukang pipa", "wastafel": "tukang pipa",
            "shower": "tukang pipa", "toilet": "tukang pipa",
            "listrik": "tukang listrik", "lampu tanam": "tukang listrik",
            "besi": "tukang besi", "railing": "tukang besi",
            "aluminium": "tukang kayu/aluminium", "pintu": "tukang kayu/aluminium",
        }
        validated = {}
        for name, info in generated.items():
            if isinstance(info, dict) and "tukang" in info and "material_utama" in info:
                if isinstance(info["material_utama"], list):
                    if info["tukang"].lower() in ("tukang umum", "umum", ""):
                        name_lower = name.lower()
                        info["tukang"] = next(
                            (v for k, v in _tukang_map_fix.items() if k in name_lower),
                            "tukang kayu/meubel"
                        )
                    validated[name] = info
        if validated:
            print(f"  AI berhasil menyusun {len(validated)} opsi custom item untuk '{room_type}'")
            return validated
        print("  (Respons AI tidak valid — pakai fallback)")
    except Exception as e:
        print(f"  (AI generate gagal: {e} — pakai fallback)")

    print(f"  (Emergency fallback: LLM gagal total untuk '{room_type}')")
    return dict(CUSTOM_ITEM_TYPES_EMERGENCY_FALLBACK)

def _ask_custom_items(room_type: str = "") -> list:
    print(f"\n[Custom Item Agent] Item custom untuk '{room_type or 'ruangan ini'}'")
    print("  (Sistem akan menyesuaikan pilihan dengan jenis ruangan yang dipilih)")
    ans = input("  Tambahkan elemen custom/tukang? (ya/tidak): ").strip().lower()
    if ans not in ("ya", "y", "yes"):
        return []

    custom_types = _generate_custom_types_llm(room_type)

    custom_list = []
    while True:
        print(f"\n  ── Opsi custom item untuk {room_type.upper() or 'ruangan ini'} ──")
        keys = list(custom_types.keys())
        for idx, k in enumerate(keys, 1):
            tukang_info = custom_types[k].get("tukang", "-")
            print(f"    {idx:2d}. {k:<30s}  [tukang: {tukang_info}]")
        print(f"    Atau ketik nama item lain yang tidak ada di daftar.")
        raw = input("  Pilih nomor atau ketik nama item (enter kosong = selesai): ").strip()
        if not raw:
            break

        chosen_key = None
        if raw.isdigit():
            idx_num = int(raw) - 1
            if 0 <= idx_num < len(keys):
                chosen_key = keys[idx_num]
                raw = chosen_key
        if chosen_key is None:
            chosen_key = next((k for k in custom_types if k in raw.lower()), None)

        if chosen_key:
            info = custom_types[chosen_key]
        else:
            info = {"tukang": "tukang umum", "material_utama": ["material sesuai kebutuhan"]}

        tukang_nama = info["tukang"]
        material_str = ", ".join(info["material_utama"])
        print(f"  Tukang yang dibutuhkan : {tukang_nama}")
        print(f"  Material utama         : {material_str}")

        try:
            budget_c = int(
                input(f"  Budget untuk '{raw}' (Rp, contoh: 1500000): ")
                .strip().replace(".", "").replace(",", "") or "0"
            )
        except ValueError:
            budget_c = 0

        ukuran = input(f"  Ukuran (opsional, contoh: 120x60x200 cm): ").strip()
        catatan_str = f"Dibuat oleh {tukang_nama} — bukan dari marketplace"

        custom_list.append({
            "name": raw,
            "tukang": tukang_nama,
            "material_utama": info["material_utama"],
            "budget_estimate": budget_c,
            "ukuran": ukuran or "-",
            "catatan": catatan_str,
        })
        print(f"  '{raw}' ditambahkan ke daftar custom.")
        if input("  Tambah item custom lagi? (ya/tidak): ").strip().lower() not in ("ya", "y"):
            break

    return custom_list

print("Custom Item Agent (full LLM) siap.")

In [ ]:
_COMMON_ROOMS = [
    "kamar tidur", "kamar mandi", "dapur", "ruang tamu", "ruang keluarga",
    "ruang makan", "ruang kerja", "balkon", "teras", "garasi", "gudang",
    "nursery", "kamar anak", "ruang bermain", "laundry",
]

def _correct_room_type(raw: str) -> str:
    raw_lower = raw.lower().strip()
    if raw_lower in _COMMON_ROOMS:
        return raw_lower
    for r in _COMMON_ROOMS:
        if r in raw_lower or raw_lower in r:
            return r
    def _sim(a, b):
        sa, sb = set(a), set(b)
        return len(sa & sb) / len(sa | sb) if sa | sb else 0
    best = max(_COMMON_ROOMS, key=lambda r: _sim(raw_lower, r))
    if _sim(raw_lower, best) > 0.4:
        return best
    try:
        _rooms_str = ", ".join(_COMMON_ROOMS)
        _prompt = (f"User mengetik nama ruangan: '{raw}'. "
                   f"Dari daftar berikut, mana yang paling cocok? {_rooms_str}. "
                   f"Jawab HANYA nama ruangan yang cocok, tanpa penjelasan.")
        _resp = groq_client.chat.completions.create(
            model=GROQ_MODEL_CANDIDATES[0],
            messages=[{"role": "user", "content": _prompt}],
            temperature=0, max_tokens=20,
        )
        _corrected = _resp.choices[0].message.content.strip().lower()
        if _corrected in _COMMON_ROOMS:
            return _corrected
    except Exception:
        pass
    return raw_lower

while True:
    raw_room = input("Jenis ruangan (contoh: kamar tidur, dapur, ruang tamu): ").strip()
    if not raw_room:
        print("  Jenis ruangan tidak boleh kosong.")
        continue
    room_type = _correct_room_type(raw_room)
    if room_type != raw_room.lower().strip():
        print(f"  ✏️  '{raw_room}' → dikoreksi menjadi: '{room_type}'")
        confirm = input(f"  Lanjut dengan '{room_type}'? (enter=ya / ketik nama lain): ").strip()
        if confirm:
            room_type = _correct_room_type(confirm) if confirm.lower() not in ("ya","y","yes") else room_type
    print(f"  ✅ Ruangan: {room_type}")
    break

def _input_float(prompt, lo=0.5, hi=30.0):
    while True:
        try:
            val = float(input(prompt).strip().replace(",", "."))
            if lo <= val <= hi:
                return val
            print(f"  Masukkan nilai antara {lo} – {hi} meter.")
        except ValueError:
            print("  Input tidak valid, masukkan angka (contoh: 3.5).")

room_length_m = _input_float("Panjang ruangan (meter, contoh: 4): ")
room_width_m  = _input_float("Lebar ruangan (meter, contoh: 3): ")
room_height_m = _input_float("Tinggi plafon (meter, contoh: 2.8): ", lo=2.0, hi=6.0)

while True:
    try:
        raw_budget = input("Budget total (Rp, contoh: 5000000): ").strip().replace(".", "").replace(",", "")
        user_budget_total = int(raw_budget)
        if user_budget_total >= 100_000:
            break
        print("  Budget minimal Rp100.000.")
    except ValueError:
        print("  Input tidak valid, masukkan angka.")

while True:
    try:
        pct_input = input("Target pemakaian budget % (contoh: 80 untuk 80%, enter = 80): ").strip() or "80"
        budget_target_pct = float(pct_input) / 100
        if 0.3 <= budget_target_pct <= 1.0:
            break
        print("  Masukkan antara 30–100.")
    except ValueError:
        print("  Input tidak valid.")

style_preference = input("Preferensi gaya/tema (opsional, contoh: minimalis, kpop, japandi): ").strip()

while True:
    try:
        floor_number = int(input("Nomor lantai (contoh: 1 untuk lantai 1, 2 untuk lantai 2): ").strip() or "1")
        if floor_number >= 1:
            break
        print("  Nomor lantai minimal 1.")
    except ValueError:
        print("  Input tidak valid, masukkan angka bulat.")

windows = []
print("\nApakah ada jendela di ruangan ini? (opsional)")
while True:
    ans_win = input("  Tambah jendela? (ya/tidak): ").strip().lower()
    if ans_win not in ("ya", "y"):
        break
    wall = ""
    while wall not in ("atas", "bawah", "kiri", "kanan"):
        wall = input("  Posisi dinding jendela (atas/bawah/kiri/kanan): ").strip().lower()
    try:
        center_m = float(input("  Posisi tengah jendela dari sudut kiri/bawah dinding (meter): ").strip() or "1.0")
        width_m  = float(input("  Lebar jendela (meter, contoh: 0.9): ").strip() or "0.9")
    except ValueError:
        center_m, width_m = 1.0, 0.9
    windows.append({"wall": wall, "center_m": round(center_m, 2), "width_m": round(width_m, 2)})
    print(f"  Jendela di dinding {wall} ditambahkan.")

user_location = input("\nLokasi kamu (kota/kecamatan, untuk cari tukang; enter=skip): ").strip()

custom_items = _ask_custom_items(room_type)

print(f"\n✅ Input diterima: {room_type} | {room_length_m}x{room_width_m}x{room_height_m}m | "
      f"Lantai {floor_number} | Budget Rp{user_budget_total:,} | Target {budget_target_pct*100:.0f}%")
print(f"   Jendela: {len(windows)} | Custom items: {len(custom_items)} | Lokasi: {user_location or '(tidak diisi)'}")

## 15. Eksekusi Graph

In [ ]:
initial_state: RoomPlannerState = {
    "room_type": room_type,
    "room_length_m": room_length_m,
    "room_width_m": room_width_m,
    "room_height_m": room_height_m,
    "style_preference": style_preference,
    "user_budget_total": user_budget_total,
    "layout_plan": {},
    "core_items": [],
    "decor_items": [],
    "recommended_items": [],
    "search_results_batch_a": {},
    "search_results_batch_b": {},
    "search_results": {},
    "selected_items": [],
    "placement": [],
    "ceiling_items": [],
    "final_floorplan_path": "",
    "retry_count": {},
    "retry_modifiers": {},
    "_needs_retry": False,
    "floor_number": floor_number,
    "windows": windows,
    "custom_items": custom_items,
    "review_results": {},
    "budget_target_pct": budget_target_pct,
    "user_location": user_location,
    "tukang_results": {},
}

final_state = None
try:
    for step_output in app.stream(initial_state):
        final_state = list(step_output.values())[0]
    print("\n=== SELESAI ===")
except Exception as e:
    print(f"\n[error] Graph berhenti di tengah jalan karena error tak terduga: {e}")
    print("  Cek pesan [warn]/[error] di atas untuk tahu node mana yang gagal. Kemungkinan "
          "besar Groq API atau kedua marketplace sama-sama total tidak bisa diakses saat ini. "
          "Coba jalankan ulang cell ini, atau lihat Bagian 7 di file penjelasan sistem.")

## 16. Tampilkan Hasil Akhir

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display, HTML

if not final_state or not final_state.get("layout_plan"):
    print("Tidak ada hasil untuk ditampilkan -- graph berhenti sebelum layout ruangan selesai dihitung.")
else:
    lp = final_state["layout_plan"]
    floor_number = final_state.get("floor_number", 1)
    print(f"1) Ruangan: {room_type} | {lp['length_m']}m x {lp['width_m']}m x {lp['height_m']}m "
          f"| Lantai {floor_number} | {lp['total_area_m2']} m2")
    windows = lp.get("windows", [])
    if windows:
        for w in windows:
            print(f"   🪟 Jendela di dinding {w['wall']}: center {w['center_m']}m, lebar {w['width_m']}m")

    print("\n2a) Furnitur dari Marketplace:")
    rows = []
    for sel in final_state.get("selected_items", []):
        p = sel["product"]
        dims = p.get("dims_cm", {})
        dim_text = f"{dims.get('width_cm','-')}x{dims.get('depth_cm','-')}x{dims.get('height_cm','-')} cm"
        fc = p.get("floor_compat", {})
        rating_raw = p.get("rating")
        rating_str = f"⭐{rating_raw:.1f}" if isinstance(rating_raw, (int, float)) and rating_raw else "—"
        platform_label = p.get("platform", "Toko")
        is_verified = p.get("verified", True)
        has_real_rating = p.get("rating") is not None
        if is_verified and has_real_rating:
            status_label = sel["status"]
            link_label   = "🛒 Beli Sekarang"
        elif is_verified and not has_real_rating:
            status_label = sel["status"]
            link_label   = "⭐ Lihat Produk Rating Terbaik"
        else:
            status_label = "⚠️ Perlu dicek — link ke halaman pencarian umum"
            link_label   = "🔍 Cari & Bandingkan"
        rows.append({
            "Kategori": sel["category"],
            "Nama Barang / Toko": p["title"][:65],
            "Dimensi (cm)": dim_text,
            "Harga Est.": p["price_text"],
            "Toko": platform_label,
            "Rating": rating_str,
            "Status": status_label,
            "Link Beli →": p.get("product_url", "-"),
            "_link_label": link_label,
        })

    if rows:
        df = pd.DataFrame(rows)
        def _make_clickable(row):
            url = row["Link Beli →"]
            if not url or url == "-":
                return "-"
            label = row.get("_link_label", "🛒 Beli Sekarang")
            return f'<a href="{url}" target="_blank" style="color:#1a73e8;font-weight:bold;">{label}</a>'
        df_display = df.copy()
        df_display["Link Beli →"] = df_display.apply(_make_clickable, axis=1)
        df_display = df_display.drop(columns=["_link_label"])
        html_table = df_display.to_html(escape=False, index=False)
        styled = f'''<style>
          table.df {{border-collapse:collapse; width:100%; font-size:13px;}}
          table.df th {{background:#1a73e8; color:white; padding:8px; text-align:left;}}
          table.df td {{padding:7px 8px; border-bottom:1px solid #e0e0e0;}}
          table.df tr:nth-child(even) {{background:#f8f9fa;}}
          table.df tr:hover {{background:#e8f0fe;}}
        </style>'''
        display(HTML(styled + html_table.replace("<table ", '<table class="df" ')))
    else:
        print("  Tidak ada barang yang berhasil direkomendasikan. Kemungkinan Groq & marketplace bermasalah saat run ini.")

    custom_items_display = final_state.get("custom_items", [])
    if custom_items_display:
        print("\n2b) Item CUSTOM (perlu tukang):")
        ci_rows = []
        for ci in custom_items_display:
            ci_rows.append({
                "Nama": ci["name"],
                "Tukang": ci["tukang"],
                "Material Utama": ", ".join(ci["material_utama"][:3]),
                "Ukuran": ci["ukuran"],
                "Estimasi Budget": f"Rp{ci.get('budget_estimate', 0):,}",
            })
        display(HTML(pd.DataFrame(ci_rows).to_html(index=False)))

    print("\n2c) Kompatibilitas Lantai:")
    for sel in final_state.get("selected_items", []):
        fc = sel["product"].get("floor_compat", {})
        wt = fc.get("weight_est_kg", 0)
        note = fc.get("note", "✅ OK")
        wt_str = f" (~{wt:.0f} kg)" if wt else ""
        print(f"  - {sel['category']}: {note}{wt_str}")

    rv = final_state.get("review_results", {})
    if rv:
        print("\n2d) Review Agent:")
        rv_rows = []
        for cat, r in rv.items():
            src_lbl    = r.get("source", "")
            rating_val = r.get("rating")
            rev_cnt    = r.get("review_count", 0)
            no_real = src_lbl in ("heuristic", "heuristic_no_data", "serpapi_unavailable",
                                  "no_data", "estimasi_ai", "llm_catalog_gshop",
                                  "llm_catalog_unverified")
            r_str = ("—" if (no_real or rating_val is None)
                     else f"⭐{float(rating_val):.1f}")
            u_str = ("—" if (no_real or not rev_cnt)
                     else f"{int(rev_cnt):,}")
            rv_rows.append({"Kategori": cat, "Rating": r_str,
                            "Jml Ulasan": u_str, "Keterangan": r.get("flag", "—")})
        display(HTML(pd.DataFrame(rv_rows).to_html(index=False)))

    total_spent = sum(parse_price(s["product"].get("price_text","0")) for s in final_state.get("selected_items",[]))
    custom_budget = sum(c.get("budget_estimate", 0) for c in final_state.get("custom_items", []))
    grand_total = total_spent + custom_budget
    budget_full = final_state.get("user_budget_total", 0)
    target_pct = final_state.get("budget_target_pct", 0.80)
    _n_selected = len(final_state.get("selected_items", []))
    _n_unverified = sum(1 for s in final_state.get("selected_items", []) if not s["product"].get("verified", True))
    print(f"\n💰 Ringkasan Budget:")
    _n_from_serpapi = sum(1 for s in final_state.get("selected_items", [])
                          if s["product"].get("verified", True) and s["product"].get("rating") is not None)
    _n_from_llm     = sum(1 for s in final_state.get("selected_items", [])
                          if s["product"].get("verified", True) and s["product"].get("rating") is None)
    if _n_unverified:
        print(f"   ⚠️ {_n_unverified}/{_n_selected} barang dengan link pencarian umum — klik untuk cari sendiri.")
    if _n_from_llm:
        print(f"   ℹ️  {_n_from_llm}/{_n_selected} barang dari katalog AI → link Tokopedia sort ulasan terbanyak.")
    if _n_from_serpapi:
        print(f"   ✅ {_n_from_serpapi}/{_n_selected} barang dari Google Shopping langsung (data real).")
    if _n_from_llm or _n_unverified:
        print(f"      Harga adalah estimasi LLM — klik link untuk lihat harga & toko terkini.")
    print(f"   Furnitur marketplace : Rp{total_spent:,}")
    if custom_budget:
        print(f"   Item custom (tukang) : Rp{custom_budget:,}")
    print(f"   TOTAL dipakai        : Rp{grand_total:,} ({grand_total/budget_full*100:.1f}% dari Rp{budget_full:,})")
    print(f"   Target pemakaian     : {target_pct*100:.0f}% = Rp{int(budget_full*target_pct):,}")
    print(f"   Sisa buffer          : Rp{budget_full - grand_total:,}")

    tukang_res = final_state.get("tukang_results", {})
    if tukang_res:
        custom_tukang = {k: v for k, v in tukang_res.items() if k != "__furnitur_custom__"}
        if custom_tukang:
            print("\n2e) Tukang untuk Item Custom:")
            t_rows = []
            for jenis, hasil in custom_tukang.items():
                for t in hasil:
                    url = t.get("maps_url", "-")
                    link_html = f'<a href="{url}" target="_blank">Maps</a>' if url and url != "-" else "-"
                    rating_str = f"{t['rating']} ⭐" if t.get("rating") else "N/A"
                    t_rows.append({
                        "Jenis Tukang": jenis,
                        "Nama": t.get("nama", "-"),
                        "Alamat": t.get("alamat", "-")[:50],
                        "Rating": rating_str,
                        "Ulasan": str(t.get("ulasan") or "-"),
                        "Telepon": t.get("telepon", "-"),
                        "Maps": link_html,
                    })
            if t_rows:
                display(HTML(pd.DataFrame(t_rows).to_html(escape=False, index=False)))
            else:
                print("  Tidak ada data tukang dari item custom.")

        fur_tukang = tukang_res.get("__furnitur_custom__", {})
        if fur_tukang:
            print("\n2f) Rekomendasi Tukang Furnitur Custom (referensi):")
            print("   (Berguna untuk custom furniture, lemari built-in, atau upgrade dari marketplace)")
            ft_rows = []
            for jenis, hasil in fur_tukang.items():
                for t in hasil:
                    url = t.get("maps_url", "-")
                    link_html = f'<a href="{url}" target="_blank">Maps</a>' if url and url != "-" else "-"
                    rating_str = f"{t['rating']} ⭐" if t.get("rating") else "N/A"
                    ft_rows.append({
                        "Kategori": jenis,
                        "Nama": t.get("nama", "-"),
                        "Alamat": t.get("alamat", "-")[:50],
                        "Rating": rating_str,
                        "Ulasan": str(t.get("ulasan") or "-"),
                        "Telepon": t.get("telepon", "-"),
                        "Maps": link_html,
                    })
            if ft_rows:
                display(HTML(pd.DataFrame(ft_rows).to_html(escape=False, index=False)))
            else:
                print("  Coba cari manual di Google Maps: 'tukang kayu furniture custom [kota kamu]'")
    else:
        print("\n2e) Tukang Finder: tidak ada lokasi diisi atau tidak ada hasil.")

    print("\n3) Peletakan barang (koordinat meter dari sudut kiri-bawah):")
    if final_state.get("placement"):
        for rect in final_state["placement"]:
            print(f"  - {rect['category']}: x={rect['x']}m, y={rect['y']}m, ukuran {rect['w']}m x {rect['d']}m")
    else:
        print("  (kosong)")

    print("\n4) Floor Plan 2D top-down (skala nyata, enhanced):")
    if final_state.get("final_floorplan_path"):
        display(IPImage(filename=final_state["final_floorplan_path"]))
    else:
        print("  Floor plan belum sempat dibuat.")

    img_3d = final_state.get("generated_image_path", "")
    if img_3d and os.path.exists(img_3d):
        print("\n5) Visualisasi Interior 3D (AI-generated — FLUX.1-schnell / SDXL):")
        print(f"   Gaya: {final_state.get('style_preference','') or 'modern minimalist'}")
        display(IPImage(filename=img_3d))
    elif not HF_TOKEN:
        print("\n5) Visualisasi Interior 3D: ⚠️  HF_TOKEN belum diset")
        print("   → https://huggingface.co/settings/tokens (gratis)")
    else:
        print("\n5) Visualisasi Interior 3D: ❌ Generate gagal (cek token / model)")

## 17a. Download Extract Kenney Furniture Kit

In [ ]:
import os, io, zipfile, requests

KENNEY_ZIP_URL = "https://kenney.nl/media/pages/assets/furniture-kit/440e0608a4-1677580847/kenney_furniture-kit.zip"
ASSET_DIR = "kenney_furniture_kit"

if not os.path.isdir(ASSET_DIR):
    print("Download Kenney Furniture Kit (CC0, ~beberapa MB)...")
    resp = requests.get(KENNEY_ZIP_URL, timeout=60)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        zf.extractall(ASSET_DIR)
    print("Selesai extract ke folder:", ASSET_DIR)
else:
    print("Asset sudah ada di folder:", ASSET_DIR, "(skip download)")

_all_png = []
for root, _dirs, files in os.walk(ASSET_DIR):
    for f in files:
        if f.lower().endswith(".png"):
            _all_png.append(os.path.relpath(os.path.join(root, f), ASSET_DIR))

_topdown_png = [p for p in _all_png if "topdown" in p.lower() or "top-down" in p.lower() or "top_down" in p.lower()]
ASSET_PNG_LIST = _topdown_png if _topdown_png else _all_png

print(f"Total PNG ditemukan: {len(_all_png)} | Dipakai (top-down jika ada): {len(ASSET_PNG_LIST)}")
print("Contoh 10 nama file pertama:", ASSET_PNG_LIST[:10])

## 17b. Mapping Kategori Sprite

In [ ]:
import difflib

FURNITURE_KEYWORDS = {
    "kasur": ["bed"], "tempat tidur": ["bed"], "ranjang": ["bed"],
    "sofa": ["sofa", "couch", "lounge"],
    "kursi": ["chair"], "kursi makan": ["chair"], "kursi kerja": ["chair", "office"],
    "meja rias": ["desk", "table", "vanity"], "meja kerja": ["desk"],
    "meja makan": ["table"], "meja kopi": ["tablecoffee", "table"],
    "meja": ["table", "desk"],
    "lemari": ["cabinet", "closet", "wardrobe"], "almari": ["cabinet", "closet"],
    "rak buku": ["bookcase", "shelf"], "rak sepatu": ["shelf", "rack"],
    "rak": ["shelf", "rack", "bookcase"],
    "tv": ["tv", "television"], "televisi": ["tv", "television"],
    "kulkas": ["fridge", "refrigerator"],
    "kompor": ["stove", "oven"], "wastafel": ["sink"], "toilet": ["toilet"],
    "bathtub": ["bathtub", "tub"], "cermin": ["mirror"],
    "lampu": ["lamp"], "tanaman": ["plant"], "pot": ["plant", "pot"],
    "karpet": ["rug", "carpet"], "bantal": ["pillow", "cushion"],
    "gorden": ["curtain"], "jam dinding": ["clock"], "bingkai": ["frame", "picture"],
    "hiasan": ["decor", "plant", "frame"], "hiasan dinding": ["frame", "picture", "clock"],
}

MANUAL_ASSET_OVERRIDE = {
}

def find_asset_for_category(category: str, png_list):
    if category in MANUAL_ASSET_OVERRIDE:
        return MANUAL_ASSET_OVERRIDE[category], "manual"

    cat_lower = category.lower()
    keywords = []
    for kata_kunci, eng_terms in FURNITURE_KEYWORDS.items():
        if kata_kunci in cat_lower:
            keywords.extend(eng_terms)

    for kw in keywords:
        for f in png_list:
            if kw in f.lower():
                return f, "exact"

    for word in cat_lower.split():
        for f in png_list:
            if word in f.lower():
                return f, "exact-word"

    guess = difflib.get_close_matches(cat_lower, [f.lower() for f in png_list], n=1, cutoff=0.3)
    if guess:
        idx = [f.lower() for f in png_list].index(guess[0])
        return png_list[idx], "fuzzy"

    return None, None

if final_state and final_state.get("placement"):
    print("Cek mapping sprite tiap barang:")
    for item in final_state["placement"]:
        found, conf = find_asset_for_category(item["category"], ASSET_PNG_LIST)
        if found:
            tag = "" if conf == "exact" else f" (~{conf}, cek manual kalau meleset)"
            print(f"  {item['category']:<28} -> {found}{tag}")
        else:
            print(f"  {item['category']:<28} -> !! TIDAK KETEMU, isi manual di MANUAL_ASSET_OVERRIDE")

## 17c. Compose Gambar Akhir

In [ ]:
from PIL import Image, ImageDraw, ImageFilter, ImageFont
import os

TARGET_FLOOR_PX   = 700
WALL_H_RATIO       = 0.30
SPRITE_SCALE_FRAC  = 0.55
MAX_SPRITE_FRAC     = 0.32
MAX_SPRITE_H_RATIO  = 0.9
OUTLINE_PX          = 3
POSTERIZE_COLORS    = 14

def _to_iso(x_m, y_m, off_x, off_y, tile_w, tile_h):
    sx = off_x + (x_m - y_m) * (tile_w / 2)
    sy = off_y + (x_m + y_m) * (tile_h / 2)
    return int(sx), int(sy)

def _draw_wall_face(draw, x0, y0, x1, y1, h_px, fill, outline, off_x, off_y, tw, th):
    p0 = _to_iso(x0, y0, off_x, off_y, tw, th)
    p1 = _to_iso(x1, y1, off_x, off_y, tw, th)
    draw.polygon([p0, p1, (p1[0], p1[1]-h_px), (p0[0], p0[1]-h_px)],
                 fill=fill, outline=outline)

def _draw_floor_tile(draw, x_m, y_m, w_m, d_m, fill, outline, off_x, off_y, tw, th):
    pts = [_to_iso(x_m,      y_m,      off_x, off_y, tw, th),
           _to_iso(x_m+w_m,  y_m,      off_x, off_y, tw, th),
           _to_iso(x_m+w_m,  y_m+d_m,  off_x, off_y, tw, th),
           _to_iso(x_m,      y_m+d_m,  off_x, off_y, tw, th)]
    draw.polygon(pts, fill=fill, outline=outline)

def cartoonify_sprite(sprite_rgba, outline_px=OUTLINE_PX, num_colors=POSTERIZE_COLORS):
    pad = outline_px + 2
    w, h = sprite_rgba.size
    padded = Image.new("RGBA", (w+pad*2, h+pad*2), (0,0,0,0))
    padded.paste(sprite_rgba, (pad, pad), sprite_rgba)
    alpha = padded.split()[3]
    dil   = alpha.filter(ImageFilter.MaxFilter(outline_px*2+1))
    ol    = Image.new("RGBA", padded.size, (25,25,25,255))
    ol.putalpha(dil)
    rgb  = padded.convert("RGB")
    post = rgb.quantize(colors=num_colors, method=Image.MEDIANCUT).convert("RGB")
    post = Image.merge("RGBA", (*post.split(), alpha))
    return Image.alpha_composite(ol, post), pad

STYLE_PALETTE = {
    "modern":       {"floor_a":"#EEE4D4","floor_b":"#E4D8C4","wall_l":"#D8CEC0","wall_b":"#E0D4C4","bg":"#C8C0B8"},
    "minimalis":    {"floor_a":"#F0EDEA","floor_b":"#E8E4DE","wall_l":"#DCDAD5","wall_b":"#E4E0D8","bg":"#CCC8C4"},
    "klasik":       {"floor_a":"#E8D8B8","floor_b":"#D8C8A8","wall_l":"#D0C0A0","wall_b":"#D8C8A8","bg":"#B8A888"},
    "scandinavian": {"floor_a":"#F0EDE8","floor_b":"#E8E4DC","wall_l":"#E0DCD5","wall_b":"#E8E4DC","bg":"#D0CCC5"},
    "industrial":   {"floor_a":"#D0CCC5","floor_b":"#C8C4BC","wall_l":"#B8B4AC","wall_b":"#C0BCA8","bg":"#A8A49C"},
    "default":      {"floor_a":"#EEE4D4","floor_b":"#E4D8C4","wall_l":"#D8CEC0","wall_b":"#E0D4C4","bg":"#C8C0B8"},
}

def _hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def compose_sprite_room_iso(state, asset_dir: str, png_list) -> Image.Image:
    lp    = state["layout_plan"]
    L, W  = lp["length_m"], lp["width_m"]
    H     = lp.get("height_m", 2.8)
    style = (state.get("style_preference") or "default").lower()
    pal_key = next((k for k in STYLE_PALETTE if k in style), "default")
    pal   = STYLE_PALETTE[pal_key]

    tile_w = (2 * TARGET_FLOOR_PX) / (L + W)
    tile_h = tile_w / 2
    wall_h_px = int(tile_w * WALL_H_RATIO * H)

    iso_corners = [_to_iso(x, y, 0, 0, tile_w, tile_h) for x, y in [(0,0),(L,0),(L,W),(0,W)]]
    min_x = min(c[0] for c in iso_corners); max_x = max(c[0] for c in iso_corners)
    min_y = min(c[1] for c in iso_corners); max_y = max(c[1] for c in iso_corners)
    PAD      = 60
    canvas_w = int(max_x - min_x) + PAD * 2
    canvas_h = int(max_y - min_y) + wall_h_px + PAD * 2 + 200
    off_x    = int(-min_x) + PAD
    off_y    = int(-min_y) + wall_h_px + PAD + 100

    floor_w_px = max_x - min_x
    max_sprite_w_px = floor_w_px * MAX_SPRITE_FRAC
    max_sprite_h_px = wall_h_px * MAX_SPRITE_H_RATIO
    longest_side_m = max(L, W)

    bg = _hex_to_rgb(pal["bg"]) + (255,)
    canvas = Image.new("RGBA", (canvas_w, canvas_h), bg)
    draw   = ImageDraw.Draw(canvas)

    wl = _hex_to_rgb(pal["wall_l"])
    step = min(0.25, W / 4)
    iy = 0.0
    while iy < W - 0.001:
        seg = min(step, W - iy)
        shade = int(215 + (iy / W) * 20)
        col = tuple(min(255, int(c * shade // 220)) for c in wl) + (255,)
        _draw_wall_face(draw, 0, iy, 0, iy+seg, wall_h_px, col, "#8A8078", off_x, off_y, tile_w, tile_h)
        iy += step

    wb = _hex_to_rgb(pal["wall_b"])
    step_x = min(0.25, L / 4)
    ix = 0.0
    while ix < L - 0.001:
        seg = min(step_x, L - ix)
        shade = int(205 + (ix / L) * 25)
        col = tuple(min(255, int(c * shade // 220)) for c in wb) + (255,)
        _draw_wall_face(draw, ix, W, ix+seg, W, wall_h_px, col, "#8A8078", off_x, off_y, tile_w, tile_h)
        ix += step_x

    WIN_C = (174, 228, 255, 210)
    for win in lp.get("windows", []):
        wall  = win.get("wall", "atas")
        ctr   = win.get("center_m", L/2)
        ww    = win.get("width_m", 0.9)
        wp_px = int(wall_h_px * 0.4)
        wbot  = int(wall_h_px * 0.3)
        if wall == "atas":
            p0 = _to_iso(ctr-ww/2, W, off_x, off_y, tile_w, tile_h)
            p1 = _to_iso(ctr+ww/2, W, off_x, off_y, tile_w, tile_h)
            pts = [(p0[0],p0[1]-wbot-wp_px),(p1[0],p1[1]-wbot-wp_px),
                   (p1[0],p1[1]-wbot),      (p0[0],p0[1]-wbot)]
            draw.polygon(pts, fill=WIN_C, outline=(74,144,217,255))
            mx = (pts[0][0]+pts[1][0])//2
            draw.line([(mx,pts[0][1]),(mx,pts[2][1])], fill=(74,144,217,180), width=2)
            draw.line([(pts[0][0],(pts[0][1]+pts[2][1])//2),
                       (pts[1][0],(pts[0][1]+pts[2][1])//2)], fill=(74,144,217,180), width=2)
        elif wall == "kiri":
            p0 = _to_iso(0, ctr-ww/2, off_x, off_y, tile_w, tile_h)
            p3 = _to_iso(0, ctr+ww/2, off_x, off_y, tile_w, tile_h)
            pts = [(p0[0],p0[1]-wbot-wp_px),(p3[0],p3[1]-wbot-wp_px),
                   (p3[0],p3[1]-wbot),      (p0[0],p0[1]-wbot)]
            draw.polygon(pts, fill=WIN_C, outline=(74,144,217,255))
            draw.line([(pts[0][0],(pts[0][1]+pts[2][1])//2),
                       (pts[1][0],(pts[0][1]+pts[2][1])//2)], fill=(74,144,217,180), width=2)

    fa = _hex_to_rgb(pal["floor_a"]) + (255,)
    fb = _hex_to_rgb(pal["floor_b"]) + (255,)
    iy_i = 0; iy = 0.0
    while iy < W - 0.001:
        ix_i = 0; ix = 0.0
        while ix < L - 0.001:
            sw = min(step_x, L-ix); sd = min(step, W-iy)
            _draw_floor_tile(draw, ix, iy, sw, sd,
                             fa if (ix_i+iy_i)%2==0 else fb,
                             "#C4B090", off_x, off_y, tile_w, tile_h)
            ix += step_x; ix_i += 1
        iy += step; iy_i += 1

    items_sorted = sorted(state["placement"], key=lambda it: it["x"] + it["y"])

    print(f"--- Room: {L}m x {W}m | skala render: {tile_w:.0f}px/m (dinamis) ---")
    for it in items_sorted:
        print(f"  {it['category']:<20} x={it['x']:.2f} y={it['y']:.2f} w={it['w']:.2f} d={it['d']:.2f}")

    n_missing = 0
    for item in items_sorted:
        asset_rel, conf = find_asset_for_category(item["category"], png_list)
        if not asset_rel:
            n_missing += 1
            cx = item["x"] + item["w"]/2; cy = item["y"] + item["d"]/2
            bx, by = _to_iso(cx, cy, off_x, off_y, tile_w, tile_h)
            pw = int(item["w"]/longest_side_m * SPRITE_SCALE_FRAC * TARGET_FLOOR_PX)
            ph = pw // 2
            draw.rectangle([bx-pw//2, by-ph-20, bx+pw//2, by],
                           fill=(170,160,150,210), outline=(60,50,40,255))
            draw.text((bx, by-ph//2-10), item["category"][:10], fill="#222", anchor="mm")
            continue
        try:
            sprite = Image.open(os.path.join(asset_dir, asset_rel)).convert("RGBA")
        except Exception as e:
            print(f"  [warn] sprite '{item['category']}': {e}")
            n_missing += 1; continue

        footprint = max(item["w"], item["d"])
        target_w  = int((footprint / longest_side_m) * SPRITE_SCALE_FRAC * TARGET_FLOOR_PX)
        target_w  = min(target_w, int(max_sprite_w_px))
        target_w  = max(target_w, 16)
        scale     = target_w / sprite.width
        new_w = max(1, int(sprite.width  * scale))
        new_h = max(1, int(sprite.height * scale))

        if new_h > max_sprite_h_px:
            shrink = max_sprite_h_px / new_h
            new_w = max(1, int(new_w * shrink))
            new_h = max(1, int(new_h * shrink))

        sprite_r = sprite.resize((new_w, new_h), Image.LANCZOS)
        sprite_c, pad = cartoonify_sprite(sprite_r)

        cx = item["x"] + item["w"]/2; cy = item["y"] + item["d"]/2
        iso_x, iso_y = _to_iso(cx, cy, off_x, off_y, tile_w, tile_h)
        px = iso_x - sprite_c.width  // 2
        py = iso_y - sprite_c.height + pad
        canvas.alpha_composite(sprite_c, (max(0, px), max(0, py)))

    if n_missing:
        print(f"  {n_missing} sprite tidak ditemukan (placeholder box).")

    title = f"{state['room_type'].title()} | {L}m × {W}m × {H}m  [{pal_key}]"
    draw.text((canvas_w//2, 14), title, fill="#333333", anchor="mt")

    bbox = canvas.getbbox()
    if bbox:
        margin = 30
        left = max(0, bbox[0] - margin)
        top = max(0, bbox[1] - margin)
        right = min(canvas.width, bbox[2] + margin)
        bottom = min(canvas.height, bbox[3] + margin)
        canvas = canvas.crop((left, top, right, bottom))

    return canvas

if final_state and final_state.get("placement"):
    room_image = compose_sprite_room_iso(final_state, ASSET_DIR, ASSET_PNG_LIST)
    out_path = "final_room_cartoon.png"
    room_image.save(out_path)
    display(room_image)
    print("Gambar disimpan:", out_path)
else:
    print("final_state belum ada / placement kosong -- jalankan graph utama (Bagian 15) dulu.")